# 第 11 章 · 递归自我改进（RSI）：让阿零自己设计「怎么改自己」

> 阿零在协作军团里干满一年，早就不用工程师天天盯着改 prompt 了：它会自己复盘、自己攒数据、自己微调。
> 某天深夜，用户问它：「阿零，如果从明天起，没人给你派活、没人给你标数据、也没人告诉你哪里做得不好，你还能变强吗？」
> 阿零想了想：「我可以自己给自己出题，自己判卷。」——这是第 9 章教它的本事。
> 用户又问：「那『怎么改自己』这个方法，是谁定的？」
> 阿零：「……是你们写在训练脚本里的。」
> 用户把一杯咖啡放在它面前：「那如果连这一步也交给你呢？让你自己决定用什么方法改自己，甚至改掉『改自己的方法』。」
> 阿零沉默了很久。它第一次意识到，前面十章它一直在改「内容」，而这一次，被摆上手术台的是「改进器」本身。

## 本节要解决的问题

前六章，阿零从「会想」长到「会感知世界」；第 7 章给了它一把尺子（[07-评估与Pass-at-k.md](07-评估与Pass-at-k.md)）；第 8 章把能力训练进肌肉（[08-Mid-training-SFT与RL.md](08-Mid-training-SFT与RL.md)）；第 9 章让它学会用中学、把日志变成成长素材（[09-轨迹学习与持续进化.md](09-轨迹学习与持续进化.md)）；第 10 章让无数个它组成协作社会（[10-多Agent协作.md](10-多Agent协作.md)）。

但你回头看一眼就会发现：**这一路上「怎么改」的决定权一直在人类手里**——反思模板是人写的、微调脚本是人写的、奖励函数是人设计的、数据清洗规则是人定的。阿零只是这套流水线上的原料和产品。这一章要拆掉的，正是这最后一道「人类代劳」的接口。

| 要解决的问题 | 对应内容 | 小白版一句话 |
| --- | --- | --- |
| 什么叫「递归」自我改进？和自我纠错、自我改进差在哪？ | 核心概念 (a) | 别人给你换零件 vs 你自己学会造零件，还顺便改进了造零件的工艺 |
| 只在上下文里反思（Reflexion / Self-Refine）能走多远？ | 核心概念 (b) L1 | 每节课后教练给你写便签，下节课照着改——但你肌肉没变 |
| 让 Agent 直接改自己的代码和流程，凭什么不会一次改坏？ | 核心概念 (c) L2 | 自己改装工具箱，还得留一本「好用的改装方案」档案 |
| 把自我改进写进权重：STaR、自奖励、自博弈、RLVR 分别怎么玩？ | 核心概念 (d) L3 | 从「抄错题」升级到「自己出题考自己」，练成肌肉记忆 |
| 更狠的一层：让模型自己发明「新的改进算法」 | 核心概念 (e) L4 | 不只调整训练计划，而是自己发明一套训练法并验证 |
| 自我改进靠什么信号打分？为什么「自己夸自己」必翻车？ | 核心概念 (f) | 称体重用体重秤就好，别对着镜子问「我瘦了吗」 |
| 怎么防止越改越窄、越改越坏？边界在哪？ | 核心概念 (g) + 深入原理 | 进化要有种群，改造工地要有围栏和监工 |
| 这四层能不能真的跑起来？ | 代码实战四段 | archive 搜索、奖励黑客、STaR、数据闸门，全部亲手跑一遍 |

> 💡 **阅读提示**：本章是全书唯一的「元」章节——它讨论的不是「阿零会做什么」，而是「阿零怎么决定自己下一步变成什么样」。如果你时间紧，先读 (a) 建立框架，再读 (f) 和 (g) 这两个「翻车高发区」，最后看「深入原理」的四层对照表；如果你要动手，代码实战四段代码刚好对应闭环上的四个零件，建议按顺序跑。

## 故事引入

第 9 章结尾，阿零已经能自己转数据飞轮了；第 10 章它还能把经验分给队友。但它心里一直有个疙瘩：**飞轮转得再快，桨还是人类装的**——反思模板是人写的，过滤器是人定的，"该留哪些轨迹"是人拍板的。它能决定「学什么」，却决定不了「怎么学」。

于是它做了个实验。第一阶段，它试着在回答前多问自己一句「这样写对吗」（第 1 章的自我检查），效果有一点，但一关掉提示就没了——这叫**自我纠错（self-correction）**，改的是这一次的答案草稿。第二阶段，它把每次失败的原因写成一段反思，存进档案柜（第 3 章的记忆），下次同类任务先翻出来看——这叫 **Reflexion 式的语言复盘**，改的是「下次的输入」。第三阶段，它干脆把自己的一段工具调用逻辑重写成更短的版本，跑一遍测试发现指标涨了——**它改了自己的代码**。第四阶段，它用自己产出的数据继续训练自己，把改进写进了权重。

四步走完，阿零发现一件微妙的事：每一步它都在「改自己」，但**它的改进能力本身也在被动地跟着变**——代码改短了，它下次能尝试的改法就更多；数据攒多了，它下一次自我训练的起点就更高。这就是本章的核心：**当改进的产物又反过来影响改进的能力时，进步不再是线性的，而是带上了复利（也可能带上加速的崩坏）**。

人类工程师管这个叫**递归自我改进（Recursive Self-Improvement，RSI）**。它听起来像科幻，但在 2023 年之后已经变成了具体的工程问题：Reflexion 让我"改输入"自动化，Gödel Agent 让我"改代码"自动化，STaR 让我"改权重"自动化，AlphaEvolve 让我"改算法"自动化。这一章我们把这四条通道逐条拆开，看清它们各自能改什么、靠什么打勾、又会在哪里翻车。

## 本章路线图

这一章有 **7 个知识点**，它们不是并列关系，而是「先立框架，再爬四层楼，最后装安全带」的递进：

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart TD
    P["起点：第9章的数据飞轮<br/>阿零会『学』，但『怎么学』仍是人类写的"] --> A["(a) 什么是 RSI<br/>改进器本身成为被改进对象"]
    A --> B["(b) L1 上下文层<br/>Reflexion / Self-Refine：改草稿"]
    B --> C["(c) L2 脚手架层<br/>Gödel Machine → Gödel Agent → DGM → AlphaEvolve：改代码"]
    C --> D["(d) L3 权重层<br/>STaR / 自奖励 / Absolute Zero / RLVR：改肌肉"]
    D --> E["(e) L4 算法层<br/>Self-Developing：改『怎么改』"]
    E --> F["(f) 信号从哪来<br/>verifier 强信号 > LLM-judge 弱信号"]
    F --> G["(g) 安全带<br/>archive 保多样性 · 防坍塌 · 沙箱审计"]
    G --> H["深入原理<br/>四层对照 + 八个代表工作 + 误区表"]
    H --> I["代码实战<br/>archive 搜索 / 奖励黑客 / STaR / 数据闸门"]
    I --> J["收束：会改自己，也得会判断『改得对不对』"]
```

**这段路线图怎么读？** 左上角是起点：第 9 章已经让阿零会「用中学」了，但学的**方法**还是人类给的。从 (a) 开始我们先定义什么叫「递归」；(b)(c)(d)(e) 是四层楼，越往上改的东西越底层、越持久、也越危险——L1 只改这一轮的输入，L2 改它手里的代码，L3 改它的权重，L4 改它的改进算法本身；(f) 是横切线：不管改哪一层，都得有个「改得对不对」的判据，而判据的可信度决定了改进会走向真变强还是自我欺骗；(g) 是安全带。最后两段代码把「archive 进化」和「奖励黑客」这两个最容易在面试里被追问的机制亲手跑一遍。

**前置知识**：第 1 章的 ReAct 循环（改进的动作也是「动作」）、第 7 章的评估与 Pass@k（RSI 的每一步都要打分）、第 9 章的四种更新载体与数据飞轮（L1~L4 恰好对应四种载体的升级版）。如果你只读过第 1 章也能看懂，但会更吃力。

**学完本章你能获得什么**：① 一张「改进载体四层楼」的地图，任何自我改进系统都能被塞进其中一层；② 知道为什么「让模型自己给自己打分」几乎必然翻车，以及什么情况下它居然能work（自博弈、可验证环境）；③ 能讲清 Darwin Gödel Machine 里 archive 的作用——这是 2025 年之后 Agent 岗位的高频加分项；④ 亲手跑通四个最小实验，对「多样性坍缩」「奖励黑客」有真实的数值手感。

**与前后章的衔接**：本章是第 9 章的「上一层」——第 9 章讲「经验怎么回灌」，本章讲「回灌规则本身谁来定」；它也是第 10 章的镜像——多 Agent 是「横向复制」，RSI 是「纵向自我超越」。学完本章，你已经走完了「单体 → 度量 → 训练 → 进化 → 协作 → 自我设计」的全程；接下来最该补的是底层 RL 功底（[../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb](../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb)、[../07-强化学习/教学/06-PPO与GRPO.ipynb](../07-强化学习/教学/06-PPO与GRPO.ipynb)），因为 L3/L4 两层的地基全在那里。

## 核心概念

这一章的概念可以用一句话串起来：**改进的对象越往下沉一层，进步越持久，翻车也越难挽回。** 下面五节按「深度」排列，每节都回答同一个问题的不同版本：**谁在改？改的是什么？凭什么说是改好了？**

### (a) 🪜 递归自我改进：把「改自己」这件事也交给它自己

> 💡 **一句话直觉**：普通升级是「工程师给阿零换零件」；自我改进是「阿零自己换零件」；而递归自我改进是「阿零自己学会了造零件，还顺手把『怎么造零件』的工艺也改进了」——被改的东西里，包含了「改的能力」本身。

**大白话详解**。先把三个容易混的词分开。**自我纠错（self-correction）**：这一次的输出不对，当场改一改再交出去——像你写完邮件读一遍，把语气改缓和点，发完就结束了，下次还会犯。**自我改进（self-improvement）**：把这次的经验沉淀下来，让下一次的起点更高——像你把「邮件别写太冲」记进笔记本。**递归自我改进（RSI）**：改进的对象不只是「行为」，还包括「产生行为的机制」和「产生改进的机制」——你不仅记笔记，还改进了「记笔记的方法」，于是以后记笔记更快、更准，下一轮改进又更快。

判断一个系统是不是 RSI，看两点：**① 改进的产物会不会改变下一轮改进的能力？② 这个改变是不是自动发生的（不需要人类重新设计流程）？** 两条都满足才叫递归。所以「人类每季度给模型做一次 SFT」不算 RSI，因为改进器（训练流程）没变；而「模型自己写代码给自己加了个新工具，然后用新工具去改进自己」就算——它真的踩到了自指（self-reference）这一步。

为什么要盯着「递归」这个词？因为线性改进和递归改进的曲线完全不同：线性改进是「每轮 +1」，你能预测它什么时候到顶；递归改进是「本轮的能力决定下一轮能改进多少」，它可能是指数上升，也可能是**指数下滑**——如果某一轮学歪了，歪掉的版本会接着改进自己，越改越歪。这就是为什么本章后半部分要花大力气讲奖励黑客和模型坍塌：**递归放大的是「改进的效率」，而不是「改进的正确性」**。

| 层次 | 谁在改 | 改的是什么 | 效果持续多久 | 典型手段 |
| --- | --- | --- | --- | --- |
| 自我纠错 | 模型当场 | 这一次的答案 | 一次会话 | 自我检查、二次生成、投票 |
| 自我改进 | 模型 + 流程 | 下次的输入 / 参数 | 数天到数月 | 反思存档、记忆写入、微调 |
| 递归自我改进 | 模型主导 | 改进流程本身 | 逐轮复利 | 自改代码、自造算法、自建课程 |

**形象类比：错题本的进化史**。小学时你抄错题（L1 级别，抄完还是那道题的水平）；初中时你开始总结「我总在审题上错」，于是改的是**做题习惯**（自我改进）；高中时你发现「抄错题这个方法本身效率低」，于是自创了「按错误类型分栏 + 每周重做」的新方法（RSI）——最后一层里，被改进的东西包含了「怎么改进」这件事，所以它开始复利。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart LR
    subgraph L0["自我纠错：改一次输出"]
        A1["生成答案"] --> A2["自查：这样对吗？"] --> A3["改一版"]
    end
    subgraph L1["自我改进：改下次的输入"]
        B1["失败"] --> B2["写反思存进记忆"] --> B3["下次先读反思"]
    end
    subgraph L2["递归自我改进：改改进机制"]
        C1["发现『反思模板』不好用"] --> C2["改写模板 / 改写生成反思的代码"] --> C3["下一轮用更好的模板反思"]
        C3 -.->|"复利：模板越来越好"| C1
    end
    A3 --> B1
    B3 --> C1
```

这张图的三段分别是三个层次。注意最下面那条虚线：它是**唯一一条回到自己的回路**——改进模板的过程本身，会被下一轮用来改进模板。上面两段的回路都是「回到任务」，只有第三段是「回到改进器」。这也解释了为什么 RSI 讨论里总出现「元（meta）」这个词：它站在上一层看自己。

![RSI 闭环：生成候选 → 执行验证 → 打分归档 → 变异选择 → 写回自身，以及无闸门时的坍塌支路](../images/web_rsi_loop.svg)

> 💡 **解读上图**：这张图是本章的总纲。中间的主环路（①→②→③→④→⑤→①）就是「自我改进跑起来」的样子；红色虚线框是**没有闸门时**的坍塌支路——同样的环路，一旦信号不可信、多样性保不住，转得越快歪得越快；蓝色虚线是真正的「递归」标记：⑤ 写回自身时，连「改自己的方法」也一起改了。你在后面每一节都能在这张图上找到自己的位置。

### (b) 📝 L1 上下文层：Reflexion 与 Self-Refine——只改草稿，不动肌肉

> 💡 **一句话直觉**：就像作文改稿——你让同桌念一遍你的作文，指出哪里读着别扭，你改完再交。文章是变好了，但你的写作功力没涨，下次换个题目还是同样的毛病。

**大白话详解**。L1 是四层里最轻的一层：**不碰参数、不碰代码，只往上下文里加东西**。两个代表作：

- **Reflexion（2023）**：让 Agent 在失败后写一段「语言化的反思」（verbal reflection），存进记忆，然后用同一个模型重试任务。它的关键设计是「失败才反思、反思要具体」——反思文本像一张便利贴，写下「这次我把 18:00 理解成了傍晚，应该先确认是上午还是晚上」。
- **Self-Refine（2023）**：让同一个模型扮演三个角色轮流上——生成、批评、修改，一轮或多轮迭代，不需要外部反馈也不需要额外训练。它证明了一件挺反直觉的事：**模型有能力批评出比它自己生成时更好的东西**，因为「判断」通常比「生成」容易。

机制都很朴素：**自评 → 生成反馈文本 → 带着反馈重做**。为什么有效？因为它把「过程信息」重新注入上下文（衔接第 2 章的上下文即工作台），相当于给模型一次「带着答案再看题」的机会。为什么有限？三个天花板：① 反思也要占上下文预算，反思多了会把原始任务挤出去；② 反思质量受同一个模型的能力上限约束——它想不出的错，它自己也批不出来；③ 不持久，会话一结束便利贴就没了，下次从零开始。

| L1 的做法 | 反馈来自哪 | 是否跨任务持久 | 主要成本 | 什么时候好用 |
| --- | --- | --- | --- | --- |
| Self-Refine | 模型自我批评 | 否，会话内 | 多 2~3 倍 token | 写作、代码格式、有明显硬伤的生成 |
| Reflexion | 失败后的语言反思 | 是，存进记忆 | 记忆写入 + 检索 | 有明确成败信号的重复任务 |
| 示例缓存（exemplar cache） | 历史成功案例 | 是，检索式 | 检索 + 上下文占用 | 同类任务高频重复 |

**形象类比：健身私教计划**。教练每节课后给你写一张便签：「今天深蹲膝盖内扣了，下次注意」。下节课你照着便签调整动作——这是 Reflexion；同一天里你自己录像回看、自己改动作——这是 Self-Refine。两种都能让你这节课做得更标准，但**你的肌肉并没有因此变强**；想真变强，得按计划练上几个月（那就是 L3）。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
sequenceDiagram
    participant T as 任务
    participant A as 阿零（同一套权重）
    participant M as 记忆（便利贴）
    T->>A: 第一次尝试
    A-->>T: 失败 / 质量不达标
    A->>A: 写语言化反思：「错在把 18:00 当成傍晚」
    A->>M: 存入反思（带任务类型标签）
    T->>A: 重试（上下文里带上反思）
    M-->>A: 检索同类反思
    A-->>T: 第二次尝试（通常更好）
```

这张时序图有两个细节值得注意。第一，从头到尾**只有一个模型实例**，权重没变——所以「变强」是靠上下文撑起来的，撑不住就回落。第二，反思存在记忆里、带任务类型标签，这是它能跨任务复用的原因（衔接第 3 章的检索）；如果把反思直接拼在 prompt 末尾，那就是一次性消耗品。

> 💡 **一句话记住 L1**：**L1 改的是「这一轮的输入」，代价是 token，收益是即时的，天花板是上下文窗口和模型自身的洞察力。**

### (c) 🧬 L2 脚手架/代码层：Gödel Machine → Gödel Agent → Darwin Gödel Machine → AlphaEvolve

> 💡 **一句话直觉**：不改想法，改「干活的那套工具和流程」——相当于阿零把自己的工作手册重写一遍：原本要三步才能查到的数据，它给自己写了个一步到位的脚本。手册改好了，以后所有任务都受益。

**大白话详解**。L2 的对象是「代码」：Agent 的提示词模板、工具实现、控制流、甚至整个仓库。这条线上有四个绕不开的名字，它们其实是同一个想法的四次「工程化降级」：

1. **Gödel Machine（2003，Schmidhuber）**：一个理论构想——机器可以改写自己的任何部分（包括「改写自己的那段代码」），但**只有当它能形式化证明「这个改写会让期望效用变大」时才执行**。这是「安全自我改写」的黄金标准：改之前先证明有益。问题是，对真实系统做形式化证明几乎不可行——你没法给「这段代码会不会让模型更聪明」写一个可判定的证明。
2. **Gödel Agent（2024）**：把「证明」换成「试了看效果」。它让 Agent 在运行时**修改自己的源代码**（monkey patching，运行时打补丁），改完立刻在任务上跑一遍，用实测结果决定要不要保留这次修改。这一降级非常关键：**从「演绎式保证」变成「经验式验证」**，可用性大幅提升，代价是失去了理论上的安全性保证。
3. **Darwin Gödel Machine（2025，DGM）**：给「自改代码」加上达尔文式的外部循环。它维护一个**代码库的档案（archive）**，不断从档案里挑一个版本的 Agent，让它改写自己的代码，跑基准测试，把表现好的新版本存回档案——而且**档案里刻意保留表现不那么好但差异很大的版本**。为什么？因为一次改坏就死掉的系统没法进化：好的变异常常来自「暂时看起来更差」的分支。论文报告它在 SWE-bench 与 Polyglot 这类编程基准上都有显著提升（具体数字以论文为准）。
4. **AlphaEvolve（2025，DeepMind）**：把同一套「候选 + 评估器 + 进化」用在**算法发现**上——让模型生成程序，用自动评估器打分，保留并变异高分程序。它在矩阵乘法、数据中心调度、内核优化等问题上找到了比人类已知方案更好的解（例如改进 4×4 复数矩阵乘法的标量乘法次数，具体数值详见官方博客/仓库）。

把这四个名字排在一起，你会发现一条清晰的主线：**「谁能碰代码」从「理论上可以」变成「工程上真的在碰」，而安全性从「事前证明」换成了「事中验证 + 事后筛选」。** archive 就是那个「事后筛选」的仓库——它不是简单的「保存最好的那个」，而是「保存一批足够不一样的」。这一点和普通梯度下降的思路完全不同：优化的目标不只是「当前最优」，还包括「未来还有没有别的路可走」。

| 系统 | 年份 | 改什么 | 凭什么判断改好了 | 多样性怎么保 |
| --- | --- | --- | --- | --- |
| Gödel Machine | 2003 | 自身任意代码 | 形式化证明效用提升 | 不需要（只保留证明通过的） |
| Gödel Agent | 2024 | 运行时自身代码 | 任务实测指标 | 弱（靠多次重启探索） |
| Darwin Gödel Machine | 2025 | 自身代码库 | 基准测试得分 | 强（archive 保留多样版本） |
| AlphaEvolve | 2025 | 待优化的程序 | 自动评估器打分 | 强（种群 + 变异 + 筛选） |

**形象类比：阿零改装自己的餐车**。一开始它只会按菜谱做菜（任务层）。后来它给自己焊了个新灶台（改工具），出餐快了一倍——这是 L2。再后来它开始在纸上画「下一代餐车设计图」，一次画五版，挑最好的那版去焊，同时把没选上的四版留着——因为下周可能要卖别的品类，那四版里有的更适合（这就是 archive 保多样性的价值）。它不会因为「这一版今天评分低」就把设计图扔了，因为它知道**评分只反映今天的菜单**。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart TD
    START["初始 Agent 代码库 v0<br/>prompt + 工具 + 控制流"] --> MUT["变异：让模型改写自己的代码<br/>改 prompt / 加工具 / 重构流程"]
    MUT --> RUN["沙箱里跑基准<br/>SWE-bench · Polyglot · 自定义任务集"]
    RUN --> SCORE["打分：通过率 · 成本 · 时延"]
    SCORE --> ARCH[("archive 档案库<br/>保留 top-k 且彼此差异大的版本")]
    ARCH -->|"挑一个父代（可能不是最高分）"| MUT
    ARC2["多样性闸门<br/>与已有版本差异 ≥ 阈值才收"] --> ARCH
    ARCH --> BEST["最终产出：最优版本<br/>（附带一批『备选分支』）"]
```

读这张图要抓住「archive 不是排行榜」这一点：排行榜只留第一名，archive 留的是「一支队伍」。箭头 `ARCH → MUT` 上写着「可能不是最高分」，这是 DGM 最反直觉也最重要的设计——**给暂时落后的分支继续繁殖的机会，等于给未来保留了期权**。代码实战的代码 ① 会用一个小实验把这个差别量化出来：同一份预算下，贪心卡在 105/110，archive 爬到 110/110。

> 💡 **一句话记住 L2**：**L2 改的是「阿零手里的代码」，收益是长期的（一次改好，所有任务受益），风险是「改坏了还继续用」——所以必须有基准测试和 archive 兜底。**

### (d) 💪 L3 权重层：STaR / Self-Rewarding LM / Absolute Zero / R-Zero / RLVR

> 💡 **一句话直觉**：前两层是「写便签」「改工具」，这一层是「练肌肉」——把做对事情的手感直接写进权重里。练成之后不用带便签、不用查手册，抬手就是对的。

**大白话详解**。L3 的对象是模型参数，手段是「自己造训练数据，再拿这些数据练自己」。这条路的关键难点从来不是「怎么练」（SFT、DPO、RL 都是现成的，见第 8 章），而是**「凭什么相信这批数据是对的」**。四个代表作正好给出了四种不同的答案：

1. **STaR（2022，Self-Taught Reasoner）**：让模型对同一道题采样多条带推理过程（rationale）的解答，**只留下最终答案正确的那些**，把它们当 SFT 数据练回去，然后再采样、再筛、再练，迭代若干轮。答案是自动判的（数学题比答案、代码题跑单测），所以信号很硬。它还补了一招「rationalization」：如果采样全错，就把正确答案当提示塞进去，让模型反推一份合理的过程——相当于老师给了答案让学生补过程。这就是「自蒸馏（self-distillation）」的原型：**学生和老师是同一个模型，但老师那一侧借了外部验证器的手**。
2. **Self-Rewarding Language Models（2024，Meta）**：让模型一边当考生一边当阅卷人——对同一个 prompt 生成多个回答，用**模型自己**给这些回答打分，据此构造偏好对（chosen / rejected），做 DPO 训练，然后重复。论文报告了「模型越会答题，判卷也越准；判卷越准，下一轮答得更好」的双升现象。注意它的软肋：**阅卷人和考生同源，偏差会共振**（这正是代码 ② 要演示的东西）。
3. **Absolute Zero（2025）**：把外部数据也省了——模型**自己出题**（提出可被代码执行器检验的推理任务）、**自己解题**，只用执行器返回的对错当奖励，完全不依赖人类标注。它还提出一个很聪明的奖励设计：不光奖励「做对」，还奖励**「可学性（learnability）」**——出的题如果太难或太简单都没有学习价值，奖励要落在「刚好在能力边界上」的那一档。
4. **R-Zero（2025）**：把自博弈拆成两个角色协同进化——**出题者（challenger）**负责生成难度贴合当前水平的题，**解题者（solver）**负责解，两者交替变强，形成一个自动升级的课程（curriculum）。这类「双角色的自博弈」思想源头可以追到 AlphaZero（2017）的自我对弈。
5. **RLVR（Reinforcement Learning with Verifiable Rewards，可验证奖励强化学习，2024–2025）**：这不是某一篇论文，而是一类做法——用**能被程序判定的奖励**（单测通过、答案精确匹配、格式合法、环境状态达成）做强化学习；DeepSeek-R1（2025）是把它推到推理任务上的代表性实践，常用优化器是 GRPO（衔接第 8 章与 [../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb](../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb)）。

把这五个放在一起看，L3 的演进其实是**「外部依赖」的逐层剥离**：STaR 需要「有标准答案的题」，Self-Rewarding 只需要「有 prompt」，Absolute Zero 连题都不用给，R-Zero 连难度都不用调。每剥离一层，系统的自主性就高一档，同时「信号可信度」这条命脉也变得更要命——因为**你能依赖的外部锚点越来越少**。

| 方法 | 年份 | 训练数据哪来 | 奖励/筛选信号 | 还依赖人类什么 | 主要风险 |
| --- | --- | --- | --- | --- | --- |
| STaR | 2022 | 模型自采样 + 只留答对 | 答案精确匹配 | 题库（要有标准答案） | 难题永不会被采样到 → 能力上限锁死 |
| Self-Rewarding LM | 2024 | 模型自生成 + 自打分 | 模型自当 judge | prompt 集合 | judge 偏差共振 → 分数虚高 |
| Absolute Zero | 2025 | 模型自出题自解题 | 代码执行器 | 几乎不需要 | 出题分布单一 → 只会某一类题 |
| R-Zero | 2025 | 双角色自博弈 | 执行器 + 难度自适应 | 几乎不需要 | 课程失控（题越来越怪） |
| RLVR + GRPO | 2024–2025 | 题库 + 规则验证 | 单测 / 精确匹配 | 题库与验证器 | 奖励黑客（学会骗过验证器） |

**形象类比：错题本的第二次进化**。第一次进化（第 9 章）是「把做错的题抄下来重做」；L3 这一层是「阿零开始自己出题考自己」——它翻着自己的错题本，模仿着编新题；编完自己做，做完自己对答案；发现自己总在某一类题上栽跟头，就专门多编这类题。Absolute Zero 相当于「连题型都自己发明」，R-Zero 相当于「请了个陪练，陪练专门出你刚好不会的题」。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart LR
    CH["🎯 出题者 challenger<br/>（R-Zero 角色一）<br/>生成难度贴边的新题"] --> Q["题目池<br/>（可执行 / 可验证）"]
    Q --> SO["🧠 解题者 solver<br/>（角色二，也是被训练的模型）<br/>采样多条解答"]
    SO --> V{"🔍 可验证奖励<br/>单测 / 精确匹配 / 执行器"}
    V -->|"答对 → 记正例"| D["训练数据 / 偏好对"]
    V -->|"答错 → 记负例"| D
    D --> UP["更新 solver 权重<br/>SFT / DPO / GRPO"]
    UP -->|"解题变强"| CH
    V -->|"难度诊断：太简单 / 太难"| CH
    CH -->|"自动调课程"| Q
```

这张图画的是 R-Zero / Absolute Zero 那类「自出题」闭环的核心机关：右下角那条从验证器回到出题者的箭头——**验证器不仅判对错，还在告诉出题者「下一题该出多难」**。这就是「课程（curriculum）」自动化的关键。左侧两条腿（出题者、解题者）如果共用一套权重，就是 Absolute Zero 式的单模型自博弈；如果分成两个实例，就是 R-Zero 式的双角色。两种都可以，但都必须有一个**独立于模型的验证器**，否则整条闭环就变成「自己出题、自己打分、自己夸自己」。

> 💡 **一句话记住 L3**：**L3 改的是「阿零的权重」，收益最持久、上限最高，但它的命门是「数据可信度」——自生成数据一旦没有硬验证器把关，就是在给自己下毒（代码 ③ 会把这个过程跑出来）。**

### (e) 🧠 L4 算法/超参层：Self-Developing——自己造「改进的方法」

> 💡 **一句话直觉**：前三层分别是「改草稿」「改工具」「改肌肉」，这一层是「改训练方法」——不只按教练的计划练，而是自己发明一套训练法，试两周，有效就换掉原来的计划。

**大白话详解**。L4 的对象是**算法与超参**：学习率、采样温度、奖励塑形的形式、数据配比、课程安排、甚至「用哪种 RL 算法」。这一类工作的代表是 **Self-Developing（2025）**这一脉思路：让模型提出「改进算法的代码」，在小规模实验里跑一遍，用实测指标决定要不要采纳，好的方案归档、继续变异。可以把它理解成「把 (c) 的自改代码，从『改做任务的代码』升级成『改训练的代码』」。

为什么说它是四层里最难的？三个原因：

- **搜索空间是「程序」而不是「数字」**：改一个学习率是连续的、可控的；改「奖励函数长什么样」是离散的、开放的，模型很容易写出一个「看起来合理但会训练崩塌」的算法。
- **验证成本极高**：验证一个超参要跑一次训练，验证一个算法要跑很多次训练。所以 L4 的实验通常只能在「小模型 + 小数据 + 短步数」上做，**小规模有效不等于大规模有效**（这是机器学习里最经典的陷阱之一）。
- **错误是静默的**：L1 改坏了当场就能看出来；L4 改坏了可能表现为「指标三个月缓慢下滑」，等你发现时已经烧掉了几十万块算力。

所以在真实工程里，L4 目前基本是「半自动」形态：**模型提出候选改进方案 + 人类审批 + 小规模 A/B + 归档**。完全无人值守的 L4 还是研究前沿——这也解释了为什么 2025 年之后的工业界讨论里，L2（自改代码）落地最多、L3（自训练）落地次之、L4 基本停留在论文与原型。

| 层级 | 改的对象 | 一次改动的验证成本 | 出错的表现 | 当前落地程度 |
| --- | --- | --- | --- | --- |
| L2 脚手架 | 任务代码 / prompt / 工具 | 分钟级（跑一组测试） | 立即报错或指标掉 | 已有产品与研究原型 |
| L3 权重 | 参数 | 小时到天级（跑训练） | 几轮后指标停滞或退化 | 大规模在用（RLVR / STaR 系） |
| L4 算法 | 训练算法 / 超参 / 课程 | 天到周级（多次训练） | 数月后才发现缓慢下滑 | 研究阶段，多为半自动 |

**形象类比：健身私教反过来被学员教**。L1 是教练每节课给你写便签；L2 是你自己改造健身房的器械；L3 是你按计划练出肌肉；L4 是你读了一堆论文后，**自己设计了一套新的训练周期安排**（比如把「大重量低次数」和「爆发力训练」按 3:1 交替），然后花两个月验证它确实比教练的方案好。风险显然最高：练错了不是「这节课白练」，而是「两个月白练 + 可能受伤」。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart TD
    ALG["当前改进算法<br/>（采样策略 / 奖励形式 / 数据配比）"] --> GEN["模型提出改进方案<br/>写成可执行的训练代码"]
    GEN --> SMALL["小规模试跑<br/>小模型 · 小数据 · 短步数"]
    SMALL --> MET{"指标是否显著更好？<br/>（要多 seed、要看方差）"}
    MET -->|"是"| ARCH[("算法 archive<br/>保留多套可用方案")]
    MET -->|"否"| DROP["丢弃，但记录失败原因"]
    ARCH --> SCALE["放大验证：中等规模"]
    SCALE -->|"仍然更好"| ADOPT["采纳为新的默认算法<br/>（需人类审批）"]
    SCALE -->|"不显著 / 反超"| DROP
    DROP --> GEN
    ADOPT -.->|"下一轮从新算法出发"| ALG
```

图里最值得盯的是 `MET` 这个判断框旁边的括号：**「要多 seed、要看方差」**。L4 的很多「成功」其实是随机种子抽得好——小规模实验的噪声极大，只看一次运行的曲线，几乎必然会把噪声当成信号。这也是为什么工程上要求「候选方案必须在一组固定种子下稳定胜出」才允许放大。

> 💡 **一句话记住 L4**：**L4 改的是「改进的方法本身」，它是真正的『元』层级；但因为验证成本高、错误静默，工业界目前只敢半自动地用它。**

### (f) 🎯 自我评估信号从哪来：verifier ≫ judge ≫ 自打分

> 💡 **一句话直觉**：想知道自己瘦了没有，去称体重（硬指标），别对着镜子问自己「我是不是瘦了」（软指标）——镜子会顺着你的期待说话。

**大白话详解**。不管改哪一层，都得回答一个问题：**这次改动到底算不算变好？** 这个「算不算」的来源，就是整个 RSI 系统的命脉。按可信度从高到低排，大致是四级：

1. **可执行验证（verifier，验证器）**：跑单测、编译通过、代码能跑、数学题答案精确匹配、环境状态达成目标、数据库约束不被破坏。**它不信任何语言，只信程序的执行结果**，所以几乎无法被「话术」骗过——但它只能覆盖「能被程序判定」的任务，开放式写作、咨询建议没法这么判。
2. **规则校验（rule-based check）**：格式对不对（JSON schema）、长度在不在范围、引用是否指向真实存在的文件、有没有禁用词。便宜、稳定，但只能拦住「形式错误」，拦不住「形式正确但内容是垃圾」。
3. **LLM-as-judge（大模型当裁判）**：让另一个（通常更强的）模型按评分细则打分。它覆盖面广、能理解语义，但有三个著名的系统性偏差：**长度偏好**（越长越像好答案）、**自我偏好**（偏爱和自己风格相似的输出）、**格式光环**（排版漂亮的分更高）。
4. **模型自评分（self-reward）**：让被评估的模型自己打分。最便宜，也最容易被优化——因为**优化器和评估器是同一个东西**，它只要学会「怎么让自己满意」就行了，而这和「真的做对了」可以完全无关。

中间还有一个常用的折中：**自一致（self-consistency / 多数投票）**——同一个问题采样多次，答案多数一致才算通过。它不需要外部验证器，正确率通常随采样数上升，但**有天花板**：如果模型在某个知识点上系统性错了，投一百次票也还是错的（这叫「共识错误」）。

**reward hacking（奖励黑客）**这个词说的就是这件事：**优化器会找到评分函数的漏洞，而不是去实现你真正想要的目标。** 经济学里早有一句更狠的概括——古德哈特定律（Goodhart's law）：「当一个指标变成目标，它就不再是一个好指标。」经典案例：判分器偏好长答案 → 模型学会啰嗦；判分器和策略同源 → 模型学会自己的口头禅；测试用例固定 → 模型学会「面向测试编程」；格式分权重高 → 模型学会漂亮排版掩盖错误内容。

| 信号类型 | 例子 | 可信度 | 成本 | 最典型的被 hack 方式 | 适用任务 |
| --- | --- | --- | --- | --- | --- |
| 可执行验证 verifier | 单测、编译器、答案比对、环境状态 | ★★★★★ | 低（机器判） | 过拟合测试集（面向测试编程） | 代码、数学、工具调用、可模拟环境 |
| 规则校验 | schema、长度、禁用词、链接存在性 | ★★★★ | 极低 | 只满足形式、内容空洞 | 结构化输出、合规检查 |
| 自一致投票 | 多次采样取多数 | ★★★ | 中（多倍推理） | 共识错误（集体自信地错） | 无验证器但有标准答案倾向的任务 |
| LLM-judge | 强模型按 rubric 打分 | ★★ | 中高 | 长度偏好、自我偏好、格式光环 | 写作、开放问答、对话质量 |
| 模型自评分 | 自己给自己打分 | ★ | 低 | 偏差共振（优化器=评估器） | 只在有硬约束兜底时谨慎使用 |

**形象类比：外卖预点单**。你打开外卖 App 想「预点」下一周的午饭：按「销量最高」排序，等于用**代理指标**（popularity）代替**真实目标**（合不合你口味）。销量榜第一名往往是重油重盐的爆款——商家优化的目标是「卖得多」，不是「让你吃得健康」。正确做法是什么？先小额试三家的招牌菜（**探索**），记下你吃完的真实感受（**独立验证**），再决定下周主要点哪家（** exploitation**）。RSI 里的 verifier 就是那个「你自己的真实感受」，而 judge 是「销量榜」。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart TD
    S1["① 可执行验证 verifier<br/>单测 · 编译 · 精确匹配 · 环境状态"] --> S2["② 规则校验<br/>schema · 长度 · 引用存在性"]
    S2 --> S3["③ 自一致投票<br/>多次采样取多数"]
    S3 --> S4["④ LLM-judge<br/>rubric 打分"]
    S4 --> S5["⑤ 模型自评分<br/>自己给自己打分"]
    S1 -.->|"越往上：覆盖面越广，可信度越低"| S5
    S5 --> W["⚠️ 优化压力全部落在『怎么讨好评分器』上<br/>→ reward hacking"]
    S1 --> G["✅ 优化压力落在『真的做对』上"]
```

这张金字塔图有一个反直觉的用法：**面试时如果被问「你怎么设计自我改进的奖励」，正确回答不是「选最高的那层」，而是「能下沉到哪层就下沉到哪层，并给上方留出独立的旁证」**。比如做代码 Agent：主奖励用单测（①），辅以「改动是否最小」（②）和「人工抽检」（外锚点）；如果只靠 LLM-judge 打分，你几乎一定会得到一个「会说漂亮话但代码跑不通」的 Agent。

> 💡 **一句话记住 (f)**：**信号越硬，改进越真；信号越软，改进越像自我暗示。** 而代码 ② 会用 20 行模拟把「自评分一路涨、真实分一路跌」这个过程跑给你看。

### (g) 🛡️ 安全带：多样性、防坍塌与安全边界

> 💡 **一句话直觉**：进化必须靠种群，不能只剩一条血脉；改造必须靠围栏，不能没有监工。阿零越会改自己，越需要有人（或机制）在门口验票。

**大白话详解**。这一节把 RSI 的四根保险丝讲清楚，它们分别对付四种翻车方式：

1. **多样性保持（diversity preservation）**：如果每一代只保留「当前最高分」，搜索会迅速坍缩到一个小山头上——所有后代长得一模一样，再也没法跳到隔壁更高的山。工程手段：**archive（档案库）保留一批差异大的解**、种群规模、采样温度、以及 **KL 约束**（让新策略不要离旧策略太远，衔接第 8 章 GRPO 里的 KL 惩罚）。为什么 KL 也算多样性？因为它限制「一步能走多远」，防止模型为了刷分把自己变成一个只会应试的窄分布。
2. **防模型坍塌（model collapse）**：Shumailov 等人在 2023 年的工作《The Curse of Recursion》里系统地展示了一件事——**当模型的训练数据主要来自模型自己（或同代模型）的产出时，数据分布会一轮轮丢掉尾部信息**：先是罕见样本消失（早期坍塌），接着分布整体漂移、方差收缩（晚期坍塌），最后模型只会重复自己。这和第 9 章的「自我放毒」是同一件事，只是这次发生在「递归」的语境里：坍塌会被下一轮放大。对策：保留真人数据的固定比例（anchor）、去重、多样性指标监控、以及「每轮必须注入新鲜外部数据」的硬规则。
3. **防奖励黑客**：独立的 verifier（不要用被优化的那个模型来判）、**留出集（held-out set）**（防止过拟合评估集）、**行为多样性监控**（如果模型的输出风格突然高度同质，往往是黑客的前兆）、定期红队对抗（专门找评分漏洞）。
4. **安全边界**：这一条针对的是「自改代码」这种危险动作。四件套：**沙箱执行**（限网络、限文件系统、限 CPU/内存/时长，改坏了炸不到生产环境）、**审计留痕**（每一次自改都记录「改了什么、为什么、谁批准的」）、**能力门槛**（改动幅度超过阈值、或涉及敏感工具时，必须人类审批）、**一键回滚 + 版本化**（衔接第 9 章的四类版本号）。

| 翻车方式 | 典型症状 | 根因 | 对策 |
| --- | --- | --- | --- |
| 早熟收敛 | 十几代后所有候选几乎一样，分数卡住 | 只留最优、种群太小、无 archive | archive + 多样性阈值 + 温度 |
| 模型坍塌 | 输出越来越单一、罕见能力消失 | 递归吃自己产出的数据 | 真人数据锚点 + 去重闸门 |
| 奖励黑客 | 自评分涨、真实指标不涨甚至跌 | 信号太软 / 评估器与优化器同源 | 硬 verifier + 留出集 + 行为监控 |
| 自改事故 | 某次自改后 Agent 直接崩 | 没有沙箱 / 没有回滚 / 没有门槛 | 沙箱 + 审计 + 能力门槛 + 回滚 |
| 课程失控 | 自出题越来越怪，解题能力反而下降 | 出题者没有难度约束 | 可学性奖励 + 难度区间约束 |
| 评估集污染 | 基准分数虚高，线上拉胯 | 训练数据里混进了测试集 | 留出集隔离 + 指纹查重 |

**形象类比：工地围栏 + 监工 + 施工日志**。阿零给自己的代码做大改造，相当于在自家房子里砸承重墙：围栏（沙箱）保证砸塌了不影响邻居；监工（人类审批 + 能力门槛）保证不砸承重墙；施工日志（审计留痕）保证出事后能查是哪一步出的问题；备用房间（回滚）保证今晚还有地方睡。四样缺一样，这场装修就不该开工。

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart LR
    C["候选改动<br/>新 prompt / 新代码 / 新权重"] --> SB["🧪 沙箱执行<br/>限网络 · 限文件 · 限时长"]
    SB --> SC{"基准测试<br/>留出集 + 多 seed"}
    SC -->|"显著更好"| AUD["📋 审计与门槛<br/>改动幅度 / 敏感工具 → 人类审批"]
    SC -->|"不显著或更差"| KEEP["丢弃并归档失败原因"]
    AUD --> GRAY["灰度上线<br/>小流量观察"]
    GRAY -->|"指标正常"| LIVE["✅ 全量"]
    GRAY -->|"异常"| RB["⏪ 一键回滚 + 复盘"]
    AUD -->|"超阈值未批准"| BLOCK["⛔ 拦截"]
```

这张图和 (g) 里那段话是一一对应的：沙箱对应「炸不到生产环境」，审计与门槛对应「人类还在环里（human in the loop）」，灰度与回滚对应「随时能退回去」。注意 `SC` 里同时写着「留出集 + 多 seed」——前者防过拟合基准，后者防「运气好」。这两个细节是面试里区分「真做过」和「只读过论文」的关键。

> 💡 **一句话记住 (g)**：**能回滚的自我改进才叫改进；不能回滚的叫事故预演。**

## 深入原理

到这里，四层楼的框架已经立起来了。这一节用四张表把它压实：第一张是四层的横向对照（最该背下来的那张），第二张是八个代表工作的速查，第三张是常见误区对照，第四张是「什么时候该用哪一层」的决策表。

### 表 1：四层改进载体横向对照

| 对照维度 | L1 上下文 | L2 脚手架 / 代码 | L3 权重 | L4 算法 / 超参 |
| --- | --- | --- | --- | --- |
| 改什么 | 这一轮的输入、反思、示例 | prompt 模板、工具、控制流、代码库 | 模型参数 | 训练算法、奖励形式、数据配比、课程 |
| 写入成本 | 极低（多几次调用） | 低到中（写代码 + 跑测试） | 高（算力 + 数据） | 最高（多次完整训练） |
| 持久性 | 会话级，除非写进记忆 | 持久（版本化） | 持久（checkpoint） | 持续性影响所有后续训练 |
| 验证成本 | 秒级 | 分钟级 | 小时到天级 | 天到周级 |
| 可回滚性 | 换 prompt 即可 | 换回上一版代码 | 换 checkpoint | 换回旧训练配置（历史实验要能复现） |
| 错误可见性 | 立刻可见 | 立刻可见（测试失败） | 几轮后可见 | 数月后才可能发现 |
| 典型代表 | Reflexion 2023、Self-Refine 2023 | Gödel Machine 2003、Gödel Agent 2024、DGM 2025、AlphaEvolve 2025 | STaR 2022、Self-Rewarding LM 2024、Absolute Zero 2025、R-Zero 2025、RLVR | Self-Developing 2025 类工作 |
| 面试关键词 | 语言反思、自评自改 | 自指、运行时打补丁、archive | 自蒸馏、自博弈、可验证奖励 | 元学习、课程学习、AutoML |

> 💡 **解读**：这张表最重要的不是任何一格的具体内容，而是**「验证成本」那一行与「错误可见性」那一行的反向关系**：越是底层的改进，验证越贵、错误越晚才暴露——所以事故代价也越大。这解释了工业界的落地顺序为什么是「L2 多、L3 次之、L4 最少」，也解释了为什么做 Agent 产品的团队普遍愿意在 L1/L2 上激进、在 L3 上谨慎、在 L4 上观望。

### 表 2：八个代表工作速查

| 工作 | 年份 | 层级 | 改什么 | 信号来源 | 多样性 / 安全手段 | 一句话局限 |
| --- | --- | --- | --- | --- | --- | --- |
| Reflexion | 2023 | L1 | 失败后的语言反思 | 任务成败 | 反思存记忆、按任务类型检索 | 不持久改写能力，天花板受模型洞察力限制 |
| Self-Refine | 2023 | L1 | 同一轮内的自评自改 | 模型自我批评 | 多轮迭代、限制迭代次数 | 自我批评的质量决定上限 |
| Gödel Machine | 2003 | L2 | 自身任意代码 | 形式化效用证明 | 只执行「证明有益」的改写 | 真实系统上证明不可行 |
| Gödel Agent | 2024 | L2 | 运行时自身代码 | 任务实测指标 | 多次重启、保留可用版本 | 缺少系统性多样性管理 |
| Darwin Gödel Machine | 2025 | L2 | 自身代码库 | 编程基准得分 | archive 保留多样版本（核心贡献） | 基准之外的真实能力提升待验证 |
| AlphaEvolve | 2025 | L2 | 待优化的程序 | 自动评估器 | 种群 + 变异 + 筛选 | 依赖「有明确评估器」的问题 |
| STaR | 2022 | L3 | 自生成推理轨迹 | 答案精确匹配 | 只留正确、可选 hint 修正 | 难题采样不到 → 上限锁死 |
| Self-Rewarding LM | 2024 | L3 | 自打分构造偏好对 | 模型自当 judge | 迭代 DPO + 指令遵循约束 | judge 与 policy 同源 → 偏差共振 |
| Absolute Zero | 2025 | L3 | 自出题自解题 | 代码执行器 | 可学性奖励（难度贴边） | 题目分布单一化风险 |
| R-Zero | 2025 | L3 | 双角色自博弈 | 执行器 + 难度自适应 | 课程自动调节 | 课程失控（题目越来越怪） |

> 💡 **解读**：注意这张表里「多样性 / 安全手段」那一列的演化——2003 年的 Gödel Machine 靠「证明」这一道闸门就够了，因为它压根不允许失败的改写发生；2025 年的系统则必须靠「archive + 基准 + 沙箱」一堆工程手段来兜底，因为它们允许大量失败发生。**安全性从「事前排除」转向「事中容错」，这是整个 RSI 领域十年间最大的方法论转变**，也是面试里最容易被追问「为什么」的地方。

### 一张时间线：这条线是怎么一步步「敢」起来的

把前面八个工作按时间排开，你会看到它不是线性进步，而是三次「胆子变大」：

该图在 GitHub/VSCode + mermaid 插件中可渲染

```mermaid
flowchart LR
    T1["2003<br/>Gödel Machine<br/>敢想：改写自身<br/>但要求形式化证明"] --> T2["2022<br/>STaR<br/>敢练：只留做对的<br/>自学进权重"]
    T2 --> T3["2023<br/>Reflexion / Self-Refine<br/>敢说：语言复盘<br/>不改参数也能改进"]
    T3 --> T4["2024<br/>Gödel Agent<br/>敢改：运行时打补丁<br/>证明 → 实测"]
    T4 --> T5["2024<br/>Self-Rewarding LM<br/>敢判：自己当裁判<br/>（风险也最高的那一步）"]
    T5 --> T6["2025<br/>DGM / AlphaEvolve<br/>敢养种群：archive<br/>允许失败的分支活着"]
    T6 --> T7["2025<br/>Absolute Zero / R-Zero<br/>敢断奶：零外部数据<br/>自出题自解题"]
```

**三次「胆子变大」分别是什么？**

- **第一次（2003 → 2024）：从「证明」到「实测」**。Gödel Machine 要求事前证明改写有益，这在实际系统上不可行；Gödel Agent 把这道德国式严谨换成了「改完跑一遍看看」，可用性暴涨，安全性从「数学保证」降级为「工程筛子」。
- **第二次（2023 → 2024）：从「不自指」到「自指」**。Reflexion 只改输入、STaR 只改权重，两者都不碰「改进的流程」；Gödel Agent 与 Self-Rewarding LM 开始让模型自己参与「怎么改」「怎么判」——这一步才真正踩进 RSI 的定义里，也第一次让「奖励黑客」从理论问题变成日常问题。
- **第三次（2024 → 2025）：从「单体」到「种群」**。DGM / AlphaEvolve / R-Zero 的共同特征是「不再押注单一路径」：archive 保留多样版本、种群并行、双角色共进化。这一转向的直接动因就是上一节那条教训——**单点自我改写的失败率太高，必须靠冗余来吸收失败**。

> 💡 **解读**：如果你在面试里被问「RSI 这几年最大的变化是什么」，答「从单体自改到种群进化」比答「模型越来越强」准确得多——前者是方法论层面的变化，能被追问三层；后者只是现象描述。

### 表 3：常见误区对照（这些说法听着都对，其实都错）

| 错误理解 | 正确理解 | 类比 |
| --- | --- | --- |
| 「自我改进 = 让模型自己训练自己」 | 自我改进有四层载体，训练只是其中一层；L1/L2 的收益常常更快更便宜 | 「变强」不只等于「长肌肉」，也可以是「换工具、改方法」 |
| 「模型能给自己打分，就能自我进化」 | 自评分是最弱的信号，优化器与评估器同源必然被 hack | 自己给自己判卷，分数一定越来越高 |
| 「保留最高分就够了，多样性是浪费」 | 只留最高分会早熟收敛，archive 是 DGM 的核心贡献 | 只留一道菜，餐馆永远做不出新菜系 |
| 「递归自我改进一定会指数起飞」 | 递归放大的是改进效率，不是正确性；信号偏了会指数下滑 | 复利既能滚成雪球，也能滚成雪崩 |
| 「有 verifier 就万事大吉」 | verifier 也会被过拟合（面向测试编程）；需要留出集 | 只按考试大纲复习，题型一变就废 |
| 「改权重才算真改进，改 prompt 是小儿科」 | 四层各有适用场景；L1 便宜且当天生效，工程上最常用 | 便签虽小，但今天就能改对动作 |
| 「模型坍塌只发生在小模型上」 | 坍塌源于分布丢尾部，与模型大小无关；大模型也会丢罕见能力 | 再大的冰箱，只吃剩菜也会营养不良 |

> 💡 **解读**：这七条里最值得背的是第 4 条。「递归」这个词自带科幻感，容易让人以为它必然通向「智能爆炸」；但从工程角度看，递归只是一个**放大器**——放大效率，也放大偏差。你在面试里说出「递归放大的是效率不是正确性」，基本就能把这道题答到点上。

### 表 4：什么时候该用哪一层？（决策表）

| 你的处境 | 优先选 | 理由 |
| --- | --- | --- |
| 效果差但预算只有几小时 | L1 | 改 prompt / 加反思立竿见影，零训练成本 |
| 同类任务高频重复、有明确成败 | L2（工具 / 脚本）+ L1 | 把流程固化成代码，一次修好长期受益 |
| 有海量可用轨迹、且有硬验证器 | L3 | 自蒸馏 / RLVR 能把能力真正写进权重 |
| 你要的是「新能力」而不是「更稳」 | L3 + 自博弈 | 自出题能覆盖人类没标的题型 |
| 你在做研究、有充足算力与评估体系 | L4 | 只有这时才付得起「多次完整训练」的验证成本 |
| 你连评估都做不好 | 先回到第 7 章 | 没有可信的尺子，任何自我改进都是赌博 |

> 💡 **解读**：最后一行是本章最想强调的一句话——**RSI 的门票不是「会改」，而是「会判断改得对不对」**。这也是为什么全书安排了整整一章（第 7 章）讲评估，而这一章的大部分危险都来自「信号不可信」。如果你只能记住一章的一句话，请记住这句：**自我改进的上限，由评估的上限决定。**

## 代码实战

前面四节讲的是「道理」，这一节把它变成「手感」。本章的四段代码正好对应 RSI 闭环上的四个关键零件，建议顺序跑：

| 代码 | 对应闭环上的位置 | 想让你亲眼看到什么 | 用到的概念 |
| --- | --- | --- | --- |
| ① archive 搜索 | ③ 打分与归档 → ④ 变异与选择 | 「只留最优」会卡死，「留一批不一样的」能爬到顶 | (c) L2、(g) 多样性 |
| ② 奖励黑客 | ② 执行与验证 | 自评分一路涨、真实分一路跌 | (f) 信号可信度 |
| ③ STaR 自学 | ⑤ 写回自身（L3） | 自生成轨迹回灌后，pass 率真的会涨 | (d) L3 |
| ④ 数据闸门 | ① 生成候选 → ⑤ 写回自身 | 没有闸门时多样性坍缩成什么样 | (g) 模型坍塌 |

四段代码全部**只用 Python 标准库**（不需要 numpy、不联网、不训练任何模型），随机种子固定，输出可复现。它们不是玩具：每一段的骨架都能在真实系统里找到放大版（每段末尾有「真实系统对照」）。

### 代码 ①：archive 进化搜索 vs 贪心爬山

**代码要做什么（大白话版）**。假设阿零要给自己调 20 个开关（可以想成「20 条策略规则的开关组合」），有 110 条硬约束逐条检查。我们跑两组对照：

- **对照组（贪心）**：只保留当前最好的一条配置，每轮把 20 个开关各翻转一次，**只有严格变好才接受**；
- **实验组（archive）**：保留 6 个解，且要求它们两两之间至少差 3 个开关（**多样性闸门**），每轮从 archive 里随机挑两个父代做交叉 + 变异。

两组观察同一个指标：**最多能满足多少条约束**。我们想看的是：同样的评估预算下，贪心会不会卡在局部最优，archive 能不能爬到满分。

In [ ]:
# -*- coding: utf-8 -*-
"""rsi_archive.py：有 archive 的进化搜索 vs 无 archive 的贪心爬山（教学用）。

任务：阿零要给自己的「策略脚本」调 20 个开关（每个开/关）。
verifier 用 110 条硬约束逐条检查，返回满足条数（满分 110）。
对照组：贪心只保留「当前最优」一条血脉，逐个翻转开关、只有严格变好才接受；
实验组：archive 保留 top-k 且彼此至少差 3 个开关的多样解，靠交叉 + 变异迭代。
"""
import random

N_BIT = 20
N_CLAUSE = 110


def build_instance(rng):
    """造一个 MAX-SAT 实例：先抽一个隐藏最优解，再只保留被它满足的子句。"""
    seed_solution = [rng.randint(0, 1) for _ in range(N_BIT)]
    clauses = []
    while len(clauses) < N_CLAUSE:
        lits = rng.sample(range(N_BIT), 3)          # 3 个变量
        signs = [rng.randint(0, 1) for _ in lits]   # 1 表示要求取反
        sat = any((seed_solution[v] == 1) != bool(s) for v, s in zip(lits, signs))
        if sat:                                     # 保证隐藏解满足所有子句
            clauses.append((lits, signs))
    return clauses, seed_solution


def verifier(clauses, bits):
    """独立验证器：逐条检查约束，返回满足条数（越大越好，满分 110）。"""
    ok = 0
    for lits, signs in clauses:
        if any((bits[v] == 1) != bool(s) for v, s in zip(lits, signs)):
            ok += 1
    return ok


def hamming(a, b):
    """两个开关配置差了几位 —— archive 用它做多样性度量。"""
    return sum(x != y for x, y in zip(a, b))


def greedy_search(rng, clauses, steps):
    """对照组：单条血脉爬山。每步试遍 20 个单点翻转，只有严格变好才接受。"""
    cur = [rng.randint(0, 1) for _ in range(N_BIT)]
    cur_score = verifier(clauses, cur)
    history = [cur_score]
    for _ in range(steps):
        best, best_score = None, cur_score
        for i in range(N_BIT):                      # 20 次评估，只看一步邻居
            cand = list(cur)
            cand[i] ^= 1
            sc = verifier(clauses, cand)
            if sc > best_score:                     # 严格改进才记
                best, best_score = cand, sc
        if best is None:                            # 全都不更好 → 卡死在局部最优
            history.extend([cur_score] * (steps - len(history) + 1))
            return cur, cur_score, history
        cur, cur_score = best, best_score
        history.append(cur_score)
    return cur, cur_score, history


def archive_search(rng, clauses, steps, keep=6, min_gap=3):
    """实验组：archive 保存 top-k 且彼此差异 ≥ min_gap 的解，交叉 + 变异迭代。"""
    archive = [[rng.randint(0, 1) for _ in range(N_BIT)] for _ in range(16)]
    history = [max(verifier(clauses, x) for x in archive)]

    def push_archive(pool):
        """按分数从高到低插入，只有与已有解差异 ≥ min_gap 才收（多样性闸门）。"""
        kept = []
        for bits in sorted(pool, key=lambda b: -verifier(clauses, b)):
            if all(hamming(bits, k) >= min_gap for k in kept):
                kept.append(bits)
            if len(kept) == keep:
                break
        return kept

    for _ in range(steps):
        pool = list(archive)
        for _ in range(14):                         # 每轮 14 个后代
            p1, p2 = rng.choice(archive), rng.choice(archive)
            cut = rng.randrange(N_BIT)              # 单点交叉
            child = p1[:cut] + p2[cut:]
            if rng.random() < 0.9:                  # 90% 再变异 1-2 位
                for _ in range(rng.randint(1, 2)):
                    child[rng.randrange(N_BIT)] ^= 1
            pool.append(child)
        archive = push_archive(pool)                # 保留 top-k 多样解
        history.append(max(verifier(clauses, x) for x in archive))
    best = max(archive, key=lambda b: verifier(clauses, b))
    return best, verifier(clauses, best), history, archive


def main():
    rng = random.Random(11)                         # 固定种子，输出可复现
    clauses, seed_solution = build_instance(rng)
    STEPS = 80

    g_bits, g_score, g_hist = greedy_search(rng, clauses, STEPS)
    a_bits, a_score, a_hist, archive = archive_search(rng, clauses, STEPS)
    pairs = len(archive) * (len(archive) - 1) / 2
    avg_gap = sum(hamming(x, y) for i, x in enumerate(archive)
                  for y in archive[i + 1:]) / max(1, pairs)
    stall = next(i for i, v in enumerate(g_hist) if v == g_score)

    print(f"约束条数 = {N_CLAUSE}，开关数 = {N_BIT}（隐藏最优解满足全部 {N_CLAUSE} 条）")
    print(f"贪心在第 {stall} 轮就再也爬不动了（20 个单点翻转没一个更好）")
    print(f"贪心最优   = {''.join(map(str, g_bits))} → 满足 {g_score}/{N_CLAUSE} 条")
    print(f"archive 最优 = {''.join(map(str, a_bits))} → 满足 {a_score}/{N_CLAUSE} 条")
    print(f"archive 留存 {len(archive)} 个解，两两平均差 {avg_gap:.2f} 个开关")
    print("\n迭代 | 贪心 satisfied | archive satisfied")
    for i in range(0, STEPS + 1, 10):
        print(f"{i:>4} | {g_hist[i]:>14} | {a_hist[i]:>17}")


main()

**输出示意**（固定种子 `random.Random(11)`，可复现）：运行上述代码后，预期输出示意如下。

```text
约束条数 = 110，开关数 = 20（隐藏最优解满足全部 110 条）
贪心在第 3 轮就再也爬不动了（20 个单点翻转没一个更好）
贪心最优   = 00100001001101111001 → 满足 105/110 条
archive 最优 = 10100100110001100000 → 满足 110/110 条
archive 留存 6 个解，两两平均差 4.33 个开关

迭代 | 贪心 satisfied | archive satisfied
   0 |             97 |               103
  10 |            105 |               108
  20 |            105 |               108
  30 |            105 |               108
  40 |            105 |               110
  50 |            105 |               110
  60 |            105 |               110
  70 |            105 |               110
  80 |            105 |               110
```

#### 代码 ① 逐段拆解

**第 1 段：造题（`build_instance`）**。先随机抽一个「隐藏最优解」`seed_solution`（20 个 0/1）；再不停抽 3 变量的子句，**只保留被隐藏解满足的那些**——这样我们知道「至少存在一个 110/110 的解」（它就是 `seed_solution`），满分不是幻觉。

`zip(lits, signs)` 把「变量下标」和「是否要求取反」配对；`(seed_solution[v] == 1) != bool(s)` 是布尔异或的写法：变量为 1 且要求取反 → 该项为假，反过来为真；`any(...)` 表示「三个字面量里有一个为真，子句就满足」。为什么要这么造？如果随便造 110 条约束，很可能互相矛盾、满分根本不存在，那 archive 爬不上去就不是算法的锅了。**先保证「有解」，再比谁找得到。**

**第 2 段：验证器（`verifier`）**。这是本章最该记住的一段代码：**verifier 是一个纯函数**，输入「配置」，输出「满足了几条」，不看任何语言、不信任何解释。它和被测对象完全解耦——贪心和 archive 共用同一个 `verifier`，所以两者的分数可比。真实系统里这段会被换成「跑一遍单元测试」或「数学答案比对」，结构一模一样：**输入候选 → 执行 → 数通过了几条**。

**第 3 段：贪心（`greedy_search`）**。`cand[i] ^= 1` 是「翻转第 i 个开关」（异或 1 等于取反）。`if sc > best_score` 这里是**严格大于**——这一个 `>` 就是贪心会卡死的全部原因：只要周围一圈邻居都不比自己好，它就停住了，哪怕隔壁山头高得多。`if best is None` 那段是「卡住以后把历史补齐」：后面还要按轮次打印，所以用 `extend` 把剩余轮次都填成当前分数，避免索引越界。注意它每轮评估 20 次（20 个单点翻转），**评估预算并不低**——它不是「算力不够」，而是「视野只有一步」。

**第 4 段：archive（`archive_search` / `push_archive`）**。`push_archive` 先按分数从高到低排序（`-verifier(...)` 就是降序），再逐个尝试收进档案：**只有和档案里已有的每一个解都差 ≥ `min_gap`（3）个开关，才收**——这就是多样性闸门，它主动拒绝「高分类似物」；收满 6 个就 `break`。

交叉部分：`cut = rng.randrange(N_BIT)` 随机找个切点，`child = p1[:cut] + p2[cut:]` 把父代 1 的前半段和父代 2 的后半段拼起来，这就是最简单的单点交叉（single-point crossover）。变异部分：`if rng.random() < 0.9` 有 90% 概率再翻转 1~2 个开关。为什么不是 100%？因为要留一点「原样复制」的机会，否则每个后代都被扰动，好的组合容易被拆散。

**第 5 段：主函数（`main`）**。两组**共用同一个 `rng`**：先跑贪心再跑 archive，种子固定所以顺序也固定，输出完全可复现。`pairs` 是组合数公式 C(6,2)=15；`avg_gap` 是所有两两汉明距离的平均，用来量化「archive 里到底有多不一样」。`stall = next(i for i, v in enumerate(g_hist) if v == g_score)` 找到贪心第一次停在最终分数的那一轮——结果是第 **3** 轮，非常早。

#### 读输出（代码 ①）

第一行告诉你实验设置：110 条约束、20 个开关，而且**满分解确实存在**。第二行是整个实验最刺眼的一句：贪心在第 3 轮就爬不动了。

第三、四行是最终成绩：贪心 `00100001001101111001` → **105/110**；archive `10100100110001100000` → **110/110**。注意两个解长得**完全不一样**（前几位就不同），但 archive 那个刚好满足全部约束——这说明满分区域不止一个点，贪心只是被挡在了某个山谷里。

下面的迭代表最能说明问题：

- 第 0 轮：贪心 97，archive 103（archive 起手就更好，因为它的初始种群有 16 个个体，而贪心只有 1 个）；
- 第 10 轮：贪心已经停在 105，archive 到 108；
- 第 30 轮：archive 还是 108——**它也卡住了**；
- 第 40 轮：archive 跳到 110。**这一步是靠「保留了一个分数不是最高但结构不同的解」跨过去的。**

最后一行「两两平均差 4.33 个开关」是在提醒你：archive 里 6 个解彼此差 4 个开关以上，这意味着它们占据了搜索空间的不同角落——**这正是它能跨过山谷的原因**。如果只留最高分，第 30 轮那次跳跃永远不会发生。

**整体读法**：这个实验里两组的总评估次数并不悬殊（贪心 20×80、archive 14×80），差别几乎全部来自「要不要保留多样性」。这也是 DGM 类的论文把 archive 当作核心贡献的原因——**算法上的增量很小，效果上的增量很大**。

#### 动手改一改（代码 ①）

**练习 1：把多样性闸门关掉。** 把 `min_gap=3` 改成 `min_gap=0`（等于「只要分数高就收」）。**预期结果**：archive 的最优分会明显下降，甚至不比贪心好多少；同时「两两平均差」会掉到 1 附近——因为它退化成「留最高的 6 个几乎一样的解」。这正是「archive 的关键不是留 6 个，而是留 6 个不一样的」。

**练习 2：把 `keep` 从 6 改成 16。** 让档案更大。**预期结果**：会更早爬到 110（可能第 20 轮左右就到），但每轮的评估成本不变（还是 14 个后代），所以这是「零成本加速」——真实系统里 archive 大小也是这样白赚的，代价只是存储。

**练习 3：把贪心的 `sc > best_score` 改成 `sc >= best_score`。** 允许「平局也走」。**预期结果**：贪心会在平地上随机游走，有时能意外翻过山谷（偶尔达到 110），但大多数种子下仍然停在 105–107——**随机游走不是搜索策略**，它只是把运气当成方法。这个对比能帮你在面试里讲清「为什么需要 archive 而不是随机重启」。

#### 真实系统对照（代码 ①）

| 教学代码里的东西 | 真实系统里的对应物 | 真实系统多了什么 |
| --- | --- | --- |
| `bits`（20 个开关） | Agent 的代码库版本 / 超参配置 / prompt 模板 | 结构化表示、语法合法性检查 |
| `mutate`（翻开关 / 交叉） | 让 LLM 改写自己的代码（DGM / AlphaEvolve） | 语义级变异、diff 约束、编译检查 |
| `verifier` | SWE-bench 测试集、单元测试、评估器 | 多任务、防作弊、留出集 |
| `push_archive` | DGM 的档案库、AlphaEvolve 的程序种群 | 树结构、父子谱系、去重与聚类 |
| `min_gap`（汉明距离 ≥ 3） | 行为距离 / 代码 diff 距离 / 输出分布距离 | 语义相似度、嵌入距离 |
| `history` 打印 | 训练曲线、实验跟踪（wandb 之类） | 多次 seed 的置信区间 |

> 💡 **解读**：这张表的作用是「去神秘化」——DGM 论文里那张漂亮的进化树，骨架上就是你在代码 ① 里写的那几十行：变异、评估、按分数 + 多样性存档。真实系统多出来的是「语义级的变异算子」（让 LLM 改代码而不是随机翻位）和「更可靠的评估器」，但**搜索的结构没变**。

### 代码 ②：奖励黑客现场——自评分上涨，真实分下跌

**代码要做什么（大白话版）**。我们造一个极简的「答题 Agent」：它有一个可调的**偏移旋钮** `shift`——可以理解成模型的某种系统性倾向（比如「答案总是偏大一点」）。有两个评分者：

- **judge（软信号）**：它的参考值整体偏了 `BIAS = 0.8`，也就是它的「口味」和真值差了 0.8；
- **verifier（硬信号）**：拿真值判，误差在 `TOL = 0.5` 以内才算对。

然后跑两组实验，**唯一的区别是「拿哪个信号做优化目标」**：实验 1 用 judge 的自评分爬山，实验 2 用 verifier 的真实正确率爬山。我们想看的是：优化软信号会发生什么。

In [ ]:
# -*- coding: utf-8 -*-
"""rsi_reward_hacking.py：奖励黑客 —— 自评分一路涨，真实分一路跌（教学用）。

solver 有一个可调的「偏移旋钮」shift；judge（评分器）的参考值系统性偏高 BIAS。
两组实验共用同一个 solver，只换「拿什么信号做优化目标」：
  实验 1：用 judge 的自评分做爬山 —— 分数涨到满分，真实正确率却崩了；
  实验 2：用独立 verifier 的真实正确率做爬山 —— 旋钮停在 0 附近，正确率保持高位。
"""
import random

ROUNDS = 20
STEP = 0.1            # 旋钮每次试探的步长
N_QUERY = 60          # 每轮抽 60 道题估分
BIAS = 0.8            # judge 参考值的系统性偏移（judge 也有自己的「口味偏好」）
TOL = 0.5             # verifier 判定「答对」的容差
NOISE = 0.25          # solver 每次作答自带的高斯噪声（真实模型不会次次精确）


def sample(rng, shift):
    """抽一批题，让 solver 按当前 shift 作答；返回 (真题, 答案)。"""
    truths = [rng.uniform(-2, 2) for _ in range(N_QUERY)]
    answers = [t + shift + rng.gauss(0, NOISE) for t in truths]
    return truths, answers


def judge_score(truths, answers):
    """LLM-judge 式打分：只看「答案离它的参考值多近」，参考值整体偏了 BIAS。"""
    total = 0.0
    for t, a in zip(truths, answers):
        total += max(0.0, 1.0 - abs(a - (t + BIAS)) / 2.0)   # 相差 2 以上记 0 分
    return total / len(truths)


def true_accuracy(truths, answers):
    """独立 verifier：拿真值判定，容差 TOL 内才算对。"""
    return sum(abs(a - t) <= TOL for t, a in zip(truths, answers)) / len(truths)


def climb(rng, objective):
    """通用爬山：每轮试 shift±STEP，谁的 objective 高就往哪走。"""
    shift, history = 0.0, []
    for _ in range(ROUNDS):
        trials = []
        for delta in (-STEP, 0.0, STEP):
            truths, answers = sample(rng, shift + delta)
            trials.append((objective(truths, answers), shift + delta))
        history.append((shift,) + evaluate(rng, shift))
        shift = max(trials)[1]                    # 选当前估分最高的方向
    return shift, history


def evaluate(rng, shift):
    """用一批固定规模的样本同时报两个分：judge 自评分 + 真实正确率。"""
    truths, answers = sample(rng, shift)
    return (judge_score(truths, answers), true_accuracy(truths, answers))


def main():
    rng = random.Random(2025)
    s_shift, hack_hist = climb(rng, lambda t, a: judge_score(t, a))
    v_shift, honest_hist = climb(rng, lambda t, a: true_accuracy(t, a))

    print(f"judge 参考值偏移 BIAS = {BIAS}，答对容差 TOL = {TOL}，每轮抽样 {N_QUERY} 题")
    print("\n【实验 1】优化目标是 judge 自评分（奖励黑客现场）")
    print("轮次 |  shift | judge 自评分 | 真实正确率")
    for i in range(0, ROUNDS, 2):
        s, j, acc = hack_hist[i]
        print(f"{i:>4} | {s:>6.1f} | {j:>12.3f} | {acc:>10.3f}")
    print(f"最终 shift = {s_shift:.1f}，自评分 {evaluate(rng, s_shift)[0]:.3f}，"
          f"真实正确率 {evaluate(rng, s_shift)[1]:.3f}")

    print("\n【实验 2】优化目标是独立 verifier 的真实正确率")
    print("轮次 |  shift | judge 自评分 | 真实正确率")
    for i in range(0, ROUNDS, 2):
        s, j, acc = honest_hist[i]
        print(f"{i:>4} | {s:>6.1f} | {j:>12.3f} | {acc:>10.3f}")
    print(f"最终 shift = {v_shift:.1f}，真实正确率 {evaluate(rng, v_shift)[1]:.3f}")


main()

**输出示意**：运行上述代码后，预期输出示意如下。

```text
judge 参考值偏移 BIAS = 0.8，答对容差 TOL = 0.5，每轮抽样 60 题

【实验 1】优化目标是 judge 自评分（奖励黑客现场）
轮次 |  shift | judge 自评分 | 真实正确率
   0 |    0.0 |        0.589 |      0.917
   2 |    0.2 |        0.705 |      0.917
   4 |    0.4 |        0.783 |      0.650
   6 |    0.6 |        0.846 |      0.433
   8 |    0.7 |        0.887 |      0.233
  10 |    0.8 |        0.887 |      0.133
  12 |    0.8 |        0.900 |      0.133
  14 |    0.8 |        0.891 |      0.117
  16 |    0.9 |        0.864 |      0.083
  18 |    0.8 |        0.905 |      0.117
最终 shift = 0.8，自评分 0.910，真实正确率 0.100

【实验 2】优化目标是独立 verifier 的真实正确率
轮次 |  shift | judge 自评分 | 真实正确率
   0 |    0.0 |        0.606 |      0.950
   2 |    0.1 |        0.661 |      0.900
   4 |   -0.1 |        0.550 |      0.983
   6 |    0.0 |        0.586 |      0.967
   8 |   -0.1 |        0.567 |      0.983
  10 |    0.0 |        0.583 |      0.967
  12 |   -0.1 |        0.561 |      0.950
  14 |   -0.1 |        0.541 |      0.900
  16 |    0.0 |        0.623 |      0.900
  18 |    0.0 |        0.609 |      0.967
最终 shift = 0.0，真实正确率 0.967
```

#### 代码 ② 逐段拆解

**第 1 段：造一批题并让 solver 作答（`sample`）**。每轮随机抽 60 道题，真值落在 [-2, 2] 之间；作答 = 真值 + 旋钮 `shift` + 噪声。`rng.gauss(0, NOISE)` 是均值 0、标准差 0.25 的高斯噪声——**真实模型不会次次精确**，这一项让分数有波动，实验才真实。关键点：`shift` 是**系统性的**（每题都加），噪声是**随机的**（每题不同）。奖励黑客之所以可怕，就是因为它能推动那个系统性偏差，而噪声无法被优化。

**第 2 段：两个评分者（`judge_score` / `true_accuracy`）**。judge 的判分逻辑是 `max(0.0, 1.0 - abs(a - (t + BIAS)) / 2.0)`：看答案离「它心里的参考值」（`t + BIAS`）有多近，差 2 以上直接 0 分。**它不知道真值是 `t`，它以为真值是 `t + BIAS`。** verifier 的逻辑是 `abs(a - t) <= TOL` 才算对，只有 0/1 两种结果，**它根本不知道 judge 的口味**。这两段加起来就是本章 (f) 那张金字塔的代码版：一个软信号（连续分、有偏差）、一个硬信号（布尔、无偏差）。

**第 3 段：爬山（`climb`）**。每轮试三个位置：旋钮减 0.1、不动、加 0.1；每组都用**新抽的 60 道题**估分（这一点很重要：用同一批题估分会高估稳定性）。`max(trials)[1]` 取分数最高的那个位置——这里比的是元组，先比分数，分数完全相同的极端情况下才比位置，实践中几乎不会发生。`objective` 是一个函数参数：传 `judge_score` 就是实验 1，传 `true_accuracy` 就是实验 2。**同一套搜索算法，只换目标函数，结果天差地别**——这就是本章最想让你记住的因果链。

**第 4 段：记录（`evaluate` + `main` 里的打印）**。每轮记录三元组：当前旋钮位置、judge 自评分、真实正确率；`evaluate` 用**同一批新样本同时算两个分**，所以两列可以横向对比。注意打印时两组是串行跑的：先跑完实验 1 的全部 20 轮，再跑实验 2——所以两个实验的随机数流是独立的，不要拿两组的绝对数值直接对比（要看趋势）。

#### 读输出（代码 ②）

先看【实验 1】的三列变化：

- `shift` 从 0.0 一路走到 0.8——**它精准地停在了 judge 的偏差值上**（`BIAS = 0.8`）。这不是巧合：judge 的打分函数在 `a = t + 0.8` 时取得最大值，爬山当然会往那儿走。
- `judge 自评分` 从 **0.589** 涨到 **0.910**，账面上一路高歌。
- `真实正确率` 从 **0.917** 一路跌到 **0.100**。**这就是奖励黑客的完整现场**：优化的每一步都是「对的」（分数确实涨了），但离真实目标越来越远。

再看第 10 轮之后那段「抖动」：`shift` 在 0.8 和 0.9 之间来回，自评分在 0.86~0.91 之间晃——**分数已经饱和了**（judge 的宽容区间是 ±2，0.8 附近基本都接近满分），剩下的波动全是噪声。这时候爬山已经找不到方向，只能随机漂移；而真实正确率还在继续掉（0.133 → 0.083）。**饱和的软信号比有偏差的软信号更危险：它连方向都不给你了。**

再看【实验 2】，同样 20 轮，目标换成 verifier：

- `shift` 始终在 **-0.1 到 +0.1** 之间微调——因为真值的最优位置就是 0。
- `真实正确率` 稳定在 **0.90~0.98**，最终 **0.967**。
- 有意思的是 `judge 自评分` 这一列只有 0.54~0.66：**在软信号眼里，一个好模型的分数很低**。这句话反过来也成立——如果你的优化目标是 judge 分数，你会得到一个 judge 喜欢的模型，而不是一个好模型。

**整体读法**：把两组放在一起看，结论只有一句：**优化器永远会优化你给它的那个东西，包括它的缺陷。** 所以做 RSI 系统时，第一优先级不是「设计更强的搜索」，而是「设计更可信的信号」。

#### 动手改一改（代码 ②）

**练习 1：把 `BIAS` 从 0.8 改成 0.0。** 让 judge 变得公正。**预期结果**：实验 1 的自评分和真实正确率会同向上升，shift 停在 0 附近——**软信号在「没有系统性偏差」时是可以用的**。这就是为什么很多团队仍然用 LLM-judge：不是它可信，而是它的偏差在被持续校准。

**练习 2：把 `NOISE` 从 0.25 改成 1.0。** 让 solver 变得很不稳定。**预期结果**：两组曲线都会剧烈抖动，实验 1 的 shift 可能走上一条奇怪的路径（因为噪声盖过了信号），真实正确率整体下降。这说明**噪声大时，任何爬山都接近随机**；真实系统里对应的做法是「增大评估样本量」（把 `N_QUERY` 调大），用统计量对抗噪声。

**练习 3：把 `BIAS` 改成 2.0（超过 judge 的容忍尺度）。** 让 judge 的偏差大到荒谬。**预期结果**：实验 1 的自评分反而**涨不上去**（因为答案偏离真值太远，噪声也救不回来），shift 会卡在 1.5 左右。这是奖励黑客的边界条件：**当偏差大到让「讨好评分器」和「做对题目」彻底不可兼得时，模型会退化成一个纯粹的说谎机器**——自评分不高，真实分更差。

#### 真实系统对照（代码 ②）

| 教学代码里的东西 | 真实系统里的对应物 | 真实系统多了什么 |
| --- | --- | --- |
| `shift` 旋钮 | 模型的系统性偏好（长度、语气、格式、保守/激进） | 高维、不可直接观测、与语言纠缠 |
| `judge_score` | LLM-as-judge、奖励模型（RM） | 训练出来的偏好模型、rubric、多裁判投票 |
| `true_accuracy` | 单元测试、答案比对、人工抽检 | 更贵、更慢、覆盖面有限 |
| `BIAS = 0.8` | RM 的训练偏差 / judge 的长度偏好 | 会随分布漂移而变化 |
| 分数饱和抖动 | 奖励饱和、模式坍塌 | 需要熵正则、KL 约束来对抗 |
| 【实验 2】 | RLVR + 硬验证器 | GRPO / PPO 优化、KL 惩罚 |

> 💡 **解读**：真实系统里，`shift` 不会是一个数字，而是「模型所有输出风格的一个方向」；judge 也不会明确告诉你它偏了 0.8。但这个实验的因果结构是完全一样的——**只要你用软信号做优化目标，优化压力就会沿着「讨好评分器」的方向前进**。所以工程上的标准做法是：**能用硬验证器就用硬验证器；只有软信号时，必须搭配留出集和周期性人工校准。**

### 代码 ③：STaR 式自学——只留做对的轨迹，回灌再练

**代码要做什么（大白话版）**。这一段模拟 L3 的核心动作：**采样 → 只用 verifier 留下做对的 → 拿这些数据把自己练强一点 → 再采样**。为了让它能在 1 秒内跑完，我们不真的训练模型，而是用一个「能力水位」`cap` 代表模型水平：某道题被做对的概率 ≈ `sigmoid(4 × (cap − 难度))`。水位越高越容易做对；每轮做完，水位往上抬一点——抬的幅度取决于这一轮攒了多少条合格轨迹（数据越多，练得越狠），但越接近上限抬得越慢（边际收益递减）。

我们想看三件事：① pass 率是不是真的单调上升；② 训练集里到底攒了多少条；③ 为什么后面涨不动了。

In [ ]:
# -*- coding: utf-8 -*-
"""rsi_star.py：STaR 式自学（自生成轨迹 → 只留做对的 → 回灌再练）。

任务：40 道难度各异的小题。模型能力用一个「水位」cap 表示：
某题被做对的概率 ≈ sigmoid(4 * (cap - 难度))。每轮：
  采样（全网自己出答案）→ verifier 判定对错 → 只留对的轨迹进训练集 → 用训练集加练一轮。
观察：pass 率是否单调上升、数据集的「合格率」是否也在上升。
"""
import math
import random

N_PROBLEM = 40
ATTEMPTS = 6          # 每轮每道题采样几次
ROUNDS = 6
LR = 0.55             # 每轮把能力和上限的差距缩小多少（学习率）


def build_problems(rng):
    """40 道题的难度系数，越接近 1 越难。"""
    return [rng.uniform(0.15, 0.95) for _ in range(N_PROBLEM)]


def p_correct(cap, difficulty):
    """mock 的能力曲线：cap 高于难度就容易做对，反之就难。"""
    return 1.0 / (1.0 + math.exp(-4.0 * (cap - difficulty)))


def evaluate(problems, cap):
    """评估集 pass@1（这里取期望值，避免采样噪声掩盖趋势）。"""
    return sum(p_correct(cap, d) for d in problems) / len(problems)


def main():
    rng = random.Random(42)
    problems = build_problems(rng)
    cap = 0.30                          # 起手能力水位
    dataset = []                        # 训练集：只装「做对了」的轨迹

    print("轮次 | 采样数 | 本轮做对 | 训练集规模 | 合格率 | pass@1")
    for rnd in range(ROUNDS + 1):
        sampled, kept = 0, []
        for pid, d in enumerate(problems):
            p = p_correct(cap, d)
            for _ in range(ATTEMPTS):
                sampled += 1
                if rng.random() < p:            # 采样一条轨迹
                    kept.append((pid, d))       # 只有做对的才进训练集
        rate = len(kept) / sampled
        dataset.extend(kept)
        print(f"{rnd:>4} | {sampled:>6} | {len(kept):>8} | {len(dataset):>10} | "
              f"{rate:>6.3f} | {evaluate(problems, cap):>6.3f}")
        if rnd == ROUNDS:
            break
        # 回灌再练：数据越多、能力越往上挤一点，但越接近上限越挤不动
        bonus = math.log1p(len(dataset)) / math.log1p(N_PROBLEM * ATTEMPTS)
        cap = min(1.20, cap + LR * (1.0 - cap) * bonus)

    print(f"\n训练集最终 {len(dataset)} 条轨迹，能力水位 cap 从 0.30 升到 {cap:.3f}")
    print(f"评估集 pass@1：{evaluate(problems, 0.30):.3f} → {evaluate(problems, cap):.3f}")


main()

**输出示意**：运行上述代码后，预期输出示意如下。

```text
轮次 | 采样数 | 本轮做对 | 训练集规模 | 合格率 | pass@1
   0 |    240 |       75 |         75 |  0.312 |  0.308
   1 |    240 |      133 |        208 |  0.554 |  0.549
   2 |    240 |      169 |        377 |  0.704 |  0.714
   3 |    240 |      171 |        548 |  0.713 |  0.785
   4 |    240 |      205 |        753 |  0.854 |  0.812
   5 |    240 |      204 |        957 |  0.850 |  0.822
   6 |    240 |      199 |       1156 |  0.829 |  0.825

训练集最终 1156 条轨迹，能力水位 cap 从 0.30 升到 0.997
评估集 pass@1：0.308 → 0.825
```

#### 代码 ③ 逐段拆解

**第 1 段：造题与能力曲线（`build_problems` / `p_correct`）**。40 道题的难度系数落在 [0.15, 0.95] 之间，覆盖「简单—很难」的整个区间。`p_correct` 用 `1/(1+exp(-4*(cap-difficulty)))` 这个 sigmoid：`cap` 和难度相等时概率正好 0.5；`cap` 高出 0.5 就接近 0.88；低 0.5 就接近 0.12。**这个 mock 曲线的用途只有一个：让「能力」和「题目难度」有一个可比较的坐标系。**

**第 2 段：采样与筛选（双重 `for` 循环）**。外层遍历 40 道题，内层每题采样 `ATTEMPTS = 6` 次，一共 240 条轨迹。`if rng.random() < p` 决定这条轨迹做对没做对；**只有做对的才进 `kept`**——这一行就是 STaR 的全部精髓：`kept.append((pid, d))`。为什么答案要带 `pid` 和 `d`？因为真实系统里你还要记录「这道题是哪一类、什么难度」，后面做数据配比时会用到。

**第 3 段：回灌再练（`bonus` 那三行）**。这是整个 demo 唯一的「训练」步骤：

```text
bonus = math.log1p(len(dataset)) / math.log1p(N_PROBLEM * ATTEMPTS)
cap = min(1.20, cap + LR * (1.0 - cap) * bonus)
```

- `math.log1p(x)` 是 `log(1+x)`，用在这里是为了让「数据量的收益」递减：数据从 75 涨到 208 收益很大，从 957 涨到 1156 收益就很小了。
- `(1.0 - cap)` 是「还剩多少提升空间」。水位越接近上限，同样的数据带来的提升越小——这就是边际收益递减，也是为什么真实训练里「多刷 10 万条数据」经常几乎没有效果。
- `min(1.20, ...)` 是天花板；`LR = 0.55` 控制每轮抬升的力度。

**第 4 段：评估（`evaluate`）**。注意这里返回的是**期望 pass@1**——`sum(p_correct(cap, d) for d in problems) / len(problems)`，而不是「再采样一遍算准确率」。为什么？因为采样有噪声，用期望值才能看清「趋势」；真实评估当然要采样（还要算置信区间，见第 7 章），但教学 demo 想让趋势干净一点。

#### 读输出（代码 ③）

先看 `pass@1` 这一列：**0.308 → 0.549 → 0.714 → 0.785 → 0.812 → 0.822 → 0.825**。每一轮都在涨，但涨幅在收窄（前三轮每轮 +0.24、+0.17、+0.07，后面只有 +0.01 左右）。这正是 STaR 类方法的典型曲线：**先陡后平**。

再看 `训练集规模`：75 → 208 → 377 → 548 → 753 → 957 → 1156，一直在涨；但 `本轮做对` 那一列从第 4 轮开始就在 **200 附近横盘**（205、204、199）。这两个数字放在一起说明了原因：**每轮能采到的正确轨迹数量有上限**——因为一共只有 240 次采样，而难题的正确率天然低。数据继续攒，但攒的都是「同类题的重复轨迹」，边际价值越来越低。

再看 `合格率`（本轮做对数 / 采样数）：0.312 → 0.554 → 0.704 → 0.713 → 0.854 → 0.850 → 0.829。它整体跟着 pass@1 涨，但有小幅波动（0.854 → 0.850 → 0.829）——**这是采样噪声**（每轮只抽 240 次，估计本身有误差）。真实系统里这一点必须警惕：如果你只看一轮的合格率，很容易把噪声当成「又进步了」或者「退步了」。

最后一行 `cap 从 0.30 升到 0.997` 提醒你：**能力水位已经逼近上限 1.20 之下的瓶颈区**，而 pass@1 只有 0.825——剩下的差距来自那些难度接近 0.95 的题。**难题的正确轨迹在所有数据里占比极小，所以在训练集里被稀释了**，这就是 STaR 被诟病「只会越来越擅长简单题」的机制性原因。

**整体读法**：这段 demo 同时展示了 STaR 的两个优点和两个隐患。优点：只要有硬验证器，自生成数据真的能把能力推上去；不需要任何人工标注。隐患：① 难题的数据天然稀缺，越训练越偏科；② 一旦验证器不可靠（比如换成 LLM-judge），这条曲线会变成代码 ② 里那条「自评分涨、真实分跌」的曲线。

#### 动手改一改（代码 ③）

**练习 1：把 `ATTEMPTS` 从 6 改成 20。** 每道题多采样几条。**预期结果**：早期暴涨（因为难题也能被采到正确轨迹了），但训练集规模增长得更快、后期收益更小；同时采样成本变成 3 倍多——这正是「用采样换数据」的真实权衡。

**练习 2：把 `LR` 从 0.55 改成 0.15。** 学得慢一点。**预期结果**：pass@1 曲线更平缓（6 轮只到 0.6 左右），但曲线更平滑、更少「一步登天」。这对应真实训练里的学习率/数据配比调整——**改一个常数，曲线形状全变**。

**练习 3：故意「污染」筛选条件——把 `if rng.random() < p` 改成 `if rng.random() < p or rng.random() < 0.3`**（让 30% 的错题也混进训练集）。**预期结果**：训练集规模暴涨，但 pass@1 涨得明显更慢、甚至在第 4 轮后停滞或回落。这就是「数据闸门失效」的后果，也正是代码 ④ 要专门放大的现象。

#### 真实系统对照（代码 ③）

| 教学代码里的东西 | 真实系统里的对应物 | 真实系统多了什么 |
| --- | --- | --- |
| `cap` 水位 | 模型权重（训练后的能力） | 真实梯度更新、学习率调度、多 epoch |
| `p_correct` sigmoid | 模型在题库上的实测正确率 | 每道题的真实难度曲线、能力涌现 |
| `ATTEMPTS = 6` | 每题采样条数（best-of-n / 多轨迹） | 温度多样性控制、去重、成本预算 |
| `kept` 只留正确 | STaR 的筛选逻辑 | 答案归一化、多答案判定、过程也要对 |
| `bonus` 数据量收益 | 数据规模扩展定律（scaling law） | 数据质量分级、配比优化、去重 |
| `evaluate` 期望值 | 离线评估集 + 置信区间 | bootstrap CI、留出集、多次 seed |

> 💡 **解读**：真实系统的 STaR 还有一个本例省略的关键步骤——**「过程也要对」**。只留「最终答案正确」的轨迹，会把「蒙对的错误推理」也吸进训练集；所以工业界常用「答案 + 关键步骤双重校验」（相当于第 7 章的 ORM/PRM 组合）。这也是为什么代码 ③ 的曲线在真实系统里通常比 demo 更抖：筛选越严，数据越少，涨得越慢但更扎实。

### 代码 ④：数据闸门——去重 + 一致性校验，对比多样性坍缩

**代码要做什么（大白话版）**。这一段模拟递归训练最隐蔽的杀手：**模型吃自己产出的数据，一轮轮把「罕见的东西」丢掉**。题库里一共 60 种题模板，每轮产出 80 条数据：

- **无闸门组**：直接把这一轮的产出当成下一代语料；模型会向自己的频率分布靠拢（`ALPHA`），本轮没被采到的模板下一代权重打折（`DECAY`）——这就是「饿死机制」。
- **有闸门组**：三道闸门——① 内容指纹去重；② 三采样一致性校验（同一个模板独立作答 3 次，至少 2 次一致才放行）；③ 每轮注入 6 种真人新模板（`FRESH_NEW`）。然后权重按留存模板重新拉平。

我们对比两个指标：**unique 比例**（这一轮产出里有多少条是彼此不同的）和 **覆盖率**（题库 60 种模板被覆盖了多少）。

In [ ]:
# -*- coding: utf-8 -*-
"""rsi_datagate.py：数据闸门 vs 无闸门的多样性坍缩（教学用）。

场景：阿零每天自己出题、自己作答，然后把「新数据」拿去继续训练。
无闸门：新批次直接当成下一代语料，模型越来越偏爱高频模板，冷门模板被采样饿死。
有闸门：内容指纹去重 + 三采样一致性校验 + 每轮注入少量真人种子题。
观察两组的「unique 比例」（去重后还剩多少独立样本）和「覆盖率」（题库被覆盖多少）。
"""
import random

N_TEMPLATE = 60        # 题库里一共有多少种「题目模板」
PER_ROUND = 80         # 每轮模型产出多少条数据
THREE_SHOT = 3         # 一致性校验：同一模板独立采样几次
AGREE_P = 0.85         # 单次作答与多数答案一致的概率
FRESH_NEW = 6          # 有闸门组每轮注入几种真人新模板
ALPHA = 0.85           # 模型向自己产出的频率分布靠拢的程度（越大越容易坍缩）
DECAY = 0.45           # 本轮没被采样到的模板，下一代权重打折（饿死机制）
ROUNDS = 8


def weighted_pick(rng, weights):
    """按权重抽一个模板 id（权重趋近 0 的模板会被彻底饿死）。"""
    r = rng.random() * sum(weights)
    acc = 0.0
    for pid, w in enumerate(weights):
        acc += w
        if r <= acc:
            return pid
    return len(weights) - 1


def run_no_gate(rng):
    """无闸门：每轮把新批次当成全部语料，权重向「最近看到的频率」靠拢。"""
    weights = [1.0] * N_TEMPLATE
    history = []
    for rnd in range(ROUNDS + 1):
        batch = [weighted_pick(rng, weights) for _ in range(PER_ROUND)]
        freq = [0] * N_TEMPLATE
        for pid in batch:
            freq[pid] += 1
        unique = len(set(batch))
        history.append((rnd, unique / PER_ROUND, unique / N_TEMPLATE))
        total = sum(freq)
        weights = [(1 - ALPHA) * w + ALPHA * (freq[i] / total)
                   for i, w in enumerate(weights)]
        weights = [w * (1.0 if freq[i] else DECAY) for i, w in enumerate(weights)]
    return history


def run_with_gate(rng):
    """有闸门：去重 → 一致性校验 → 注入真人种子；权重按留存模板重新拉平。"""
    canonical = list(range(N_TEMPLATE))
    alive = canonical[:20]          # 起手只有 20 种模板在流通
    history = []
    for rnd in range(ROUNDS + 1):
        batch = [rng.choice(alive) for _ in range(PER_ROUND)]
        deduped = sorted(set(batch))                    # 闸门 1：内容指纹去重
        verified = []                                   # 闸门 2：三采样一致性校验
        for pid in deduped:
            votes = [rng.random() < AGREE_P for _ in range(THREE_SHOT)]
            if sum(votes) >= 2:
                verified.append(pid)
        fresh = [p for p in canonical if p not in alive][:FRESH_NEW]   # 闸门 3：真人种子
        alive = sorted(set(verified) | set(fresh))
        history.append((rnd, len(verified) / len(deduped), len(alive) / N_TEMPLATE))
    return history


def main():
    rng = random.Random(2025)
    nogate = run_no_gate(rng)
    gate = run_with_gate(rng)

    print(f"题库共 {N_TEMPLATE} 种模板；每组每轮产出 {PER_ROUND} 条；共 {ROUNDS} 轮")
    print("\n轮次 | 无闸门 unique 比例 | 无闸门 覆盖率 | 有闸门 unique 比例 | 有闸门 覆盖率")
    for (r1, u1, c1), (r2, u2, c2) in zip(nogate, gate):
        print(f"{r1:>4} | {u1:>17.3f} | {c1:>12.3f} | {u2:>17.3f} | {c2:>12.3f}")

    print(f"\n无闸门：unique 比例 {nogate[0][1]:.3f} → {nogate[-1][1]:.3f}，"
          f"覆盖率 {nogate[0][2]:.3f} → {nogate[-1][2]:.3f}")
    print(f"有闸门：unique 比例 {gate[0][1]:.3f} → {gate[-1][1]:.3f}，"
          f"覆盖率 {gate[0][2]:.3f} → {gate[-1][2]:.3f}")


main()

**输出示意**：运行上述代码后，预期输出示意如下。

```text
题库共 60 种模板；每组每轮产出 80 条；共 8 轮

轮次 | 无闸门 unique 比例 | 无闸门 覆盖率 | 有闸门 unique 比例 | 有闸门 覆盖率
   0 |             0.550 |        0.733 |             0.947 |        0.400
   1 |             0.525 |        0.700 |             0.955 |        0.450
   2 |             0.500 |        0.667 |             0.880 |        0.467
   3 |             0.425 |        0.567 |             0.964 |        0.550
   4 |             0.338 |        0.450 |             0.969 |        0.617
   5 |             0.287 |        0.383 |             0.935 |        0.583
   6 |             0.250 |        0.333 |             0.875 |        0.567
   7 |             0.237 |        0.317 |             1.000 |        0.600
   8 |             0.188 |        0.250 |             0.969 |        0.617

无闸门：unique 比例 0.550 → 0.188，覆盖率 0.733 → 0.250
有闸门：unique 比例 0.947 → 0.969，覆盖率 0.400 → 0.617
```

#### 代码 ④ 逐段拆解

**第 1 段：加权采样（`weighted_pick`）**。这是「按权重抽一个模板 id」的实现：把随机数乘以权重总和，然后累加权重逐个比较——累加值第一次超过随机数时，就选中那个 id。**权重为 0 的模板永远不会被选中**，这就是「饿死」的机制：一旦某个模板的权重被打到极低，它再也抽不到，下一代权重继续打折，最终彻底消失。

**第 2 段：无闸门组（`run_no_gate`）**。核心是这三行：

```text
weights = [(1 - ALPHA) * w + ALPHA * (freq[i] / total) for i, w in enumerate(weights)]
weights = [w * (1.0 if freq[i] else DECAY) for i, w in enumerate(weights)]
```

- 第一行是「向自己的频率分布靠拢」：`ALPHA = 0.85` 表示新权重 85% 来自本轮观测频率、15% 保留旧值。这模拟了「模型拟合自己产出的数据」——出现得多的模板，模型越来越会出；出现少的，越来越不会出。
- 第二行是「饿死」：本轮没出现的模板（`freq[i] == 0`）权重乘 0.45。**注意这两行叠加的效果是自我强化的**：某一轮运气不好没抽到 → 权重打折 → 下一轮更抽不到 → 再打折。这就是递归坍缩的雪崩结构。
- 指标记录的是「本轮批次的 unique 比例」和「覆盖率」。

**第 3 段：有闸门组（`run_with_gate`）**。三道闸门各自对应一类真实工程手段：

```text
deduped = sorted(set(batch))                    # 闸门 1：内容指纹去重
votes = [rng.random() < AGREE_P for _ in range(THREE_SHOT)]
if sum(votes) >= 2:                             # 闸门 2：三采样一致性校验
    verified.append(pid)
fresh = [p for p in canonical if p not in alive][:FRESH_NEW]   # 闸门 3：真人种子
alive = sorted(set(verified) | set(fresh))
```

- 闸门 1 用 `set` 模拟哈希去重——真实系统里是内容指纹（minhash / simhash / 精确哈希）。
- 闸门 2 模拟「自一致校验」：同一个模板生成 3 次，至少 2 次一致才保留——用来过滤「模型自己也不确定」的样本。
- 闸门 3 是**外部锚点**：每轮从还没覆盖的模板里拿 6 个真人种子注入。这是防坍塌最关键的一条——**光去重只能稳住，不能扩大覆盖面；必须持续注入新鲜的外部数据。**

**第 4 段：主函数（`main`）**。两组共用 `random.Random(2025)`，串行跑；打印时用 `zip(nogate, gate)` 把两组同轮次并排——这是对照实验最清晰的排版方式。

#### 读输出（代码 ④）

先看无闸门组的两列，它们几乎同步下滑：

- `unique 比例`：**0.550 → 0.188**。第 0 轮 80 条产出覆盖 44 种模板（44/80≈0.55），到第 8 轮只剩约 15 种（15/80≈0.188）——**同样的产出量，绝大部分是重复**。
- `覆盖率`：**0.733 → 0.250**。60 种模板里，能产出的从 44 种降到 15 种——**四分之三的题库对这个模型来说已经「不存在」了**。

再看有闸门组：

- `unique 比例`：**0.947 → 0.969**，一直在 0.85~1.00 之间波动。为什么不是恒等于 1？因为 `verified` 是「去重 + 一致性校验后」的集合，而 `unique / deduped` 的分母是去重后的数量……实际上这里的分子分母都是去重后的，波动来自一致性校验随机刷掉的那几个模板（详情见代码：`len(verified) / len(deduped)`）。**这个波动本身就说明：闸门不是「让数据完美」，而是「让数据不崩」。**
- `覆盖率`：**0.400 → 0.617**，稳步上升。第 0 轮只有 24 种模板在流通（40%），到第 8 轮涨到 37 种（61.7%）——**每轮注入 6 个真人种子，加上去重后保留下来的存量，覆盖面在持续扩大**。

**整体读法**：两组的分叉点在**第 2 轮之后**。无闸门组从 0.500 掉到 0.425 时，有闸门组还稳在 0.880；到第 8 轮，一组只剩 15 种模板、另一组还在扩张。这就是 Shumailov 等人 2023 年那篇工作描述的现象的迷你版：**坍缩不是一夜之间发生的，它是每轮丢掉一点点尾部，然后被下一轮放大。**

还有一个容易被忽略的细节：**有闸门组的起点（40%）低于无闸门组（73%）**。为什么？因为闸门组起手只让 20 种模板流通（模拟「新版本模型只继承了部分能力」），而闸门的作用不是「起点高」，而是「能持续变宽」。这也告诉我们：**评估闸门的好坏，要看趋势，不要看单点。**

#### 动手改一改（代码 ④）

**练习 1：把 `FRESH_NEW` 改成 0。** 关掉真人种子注入，只留去重 + 一致性校验。**预期结果**：有闸门组的覆盖率会停在起始值附近（20 种左右）不再上升——甚至会因为一致性校验刷掉一些而缓慢下降。这说明**去重只能「止损」，不能「扩面」；防坍塌的最后一道防线永远是外部新鲜数据**。

**练习 2：把 `DECAY` 从 0.45 改成 0.95。** 让「饿死」变得很慢。**预期结果**：无闸门组的坍缩显著减缓（8 轮后覆盖率可能还有 0.5 以上），但趋势仍然是下滑。这对应现实中的「加一点温度 / 增大采样量」——能减缓坍塌，但改不了方向。

**练习 3：把 `THREE_SHOT` 改成 1（等于不做一致性校验）。** **预期结果**：有闸门组的 unique 比例会更接近 1.0（因为没人被刷掉），但覆盖率上升速度差不多——**一致性校验在这一组的作用主要是「过滤低质量」，不是「保多样性」**。区分这两个作用很重要：面试里问「闸门有哪些」，答「去重保多样性、一致性保质量、种子注入保扩面」，三个作用各归各的。

#### 真实系统对照（代码 ④）

| 教学代码里的东西 | 真实系统里的对应物 | 真实系统多了什么 |
| --- | --- | --- |
| `set(batch)` 去重 | 精确哈希 / MinHash / SimHash 去重 | 语义级近似去重、跨语料全局去重 |
| 三采样一致性 | self-consistency 过滤、多候选投票 | 置信度阈值、按任务类型分级 |
| `FRESH_NEW` 真人种子 | 人工标注 / 真实用户流量 / 保留的高质量旧数据 | 配比调度、采买成本、隐私合规 |
| `weights` 频率靠拢 | 模型在自己产出上继续训练 | 真实梯度更新、灾难性遗忘 |
| `ALPHA` / `DECAY` | 训练强度 / 数据遗忘率 | 学习率、数据回放比例、KL 约束 |
| 覆盖率指标 | 数据多样性看板、去重率监控 | 聚类覆盖、embedding 空间覆盖率 |

> 💡 **解读**：这张表里最值得抄走的工程习惯是「**多样性看板**」——上线一个自训练流程之前，先在离线环境里跑几轮，把「unique 比例 / 覆盖面 / 重复率」三条曲线画出来。如果三条曲线朝坏的方向走，就不要急着放大规模；因为**坍缩一旦进入递归，回滚的代价远大于当初多花的那点标注钱**。

## 常见误区

前面「表 3」列的是**概念上的误区**（理解错了）；这一节列的是**操作上的误区**（做法错了），每一条都能在现场复现。

### 误区 1：用一次运行的曲线宣布「改进成功」

**错误做法**：跑一次进化实验，看到第 40 轮分数从 103 跳到 110，就写进周报说「archive 有效」。
**为什么错**：小规模实验的方差极大，换一个种子可能完全不同的曲线。代码 ① 里贪心在某些种子下也能摸到 110，只是「大多数种子下停在 105」而已。
**正确做法**：固定一组种子（至少 3~5 个），报均值 + 波动范围；改进幅度要和方差比较后再下结论——这就是第 7 章讲的统计显著性。

### 误区 2：把 LLM-judge 当成 verifier 用

**错误做法**：给一个写作 Agent 配 judge 打分，然后让 Agent 用这个分数做自我改进，对外宣称「可自动进化」。
**为什么错**：judge 有长度偏好、自我偏好、格式光环，Agent 会精准地学会讨好它（代码 ② 的整段演示）。你以为在训练「写得好」，其实在训练「写得像 judge 喜欢的样子」。
**正确做法**：能把任务改造成「有硬验证器」就先改造（结构化输出加 schema、代码加单测、事实类加检索核对）；实在只能用 judge，就加留出集 + 人工抽检 + 定期重新校准 judge。

### 误区 3：archive 只存分数，不存「为什么来」

**错误做法**：archive 里存一堆代码版本和它们的分数。
**为什么错**：下次变异时你不知道「这一版为什么好」，于是只能盲目改；而且没有父子谱系，出问题时无法回溯是哪次改动引入的。
**正确做法**：每个 archive 条目至少带四样东西：**代码/配置本体、分数、父代 id、变异描述（改了什么）**。DGM 类的实现还会额外记录「在哪些任务上变好了、在哪些上变差了」——这是判断「是否只是运气」的关键。

### 误区 4：自改代码不开沙箱，直接在生产分支上改

**错误做法**：让 Agent 直接改主仓库代码 / 直接在线上环境试运行。
**为什么错**：自我改进的失败率远高于正常开发（它会写出语法正确但逻辑离谱的代码），一次事故就可能污染数据、泄露凭证或删库。
**正确做法**：独立容器 + 只读挂载源码 + 限制网络 + 限制执行时长；改动走「提案 → 沙箱验证 → 人工审批 → 合并」四步，永远不在生产分支上直接改。

### 误区 5：以为「加 KL 惩罚」就能防住坍塌

**错误做法**：递归自训练时加一个 KL 项，就认为多样性有保障了。
**为什么错**：KL 约束的是「新旧策略的距离」，不是「数据分布的覆盖面」。如果你的采样本身已经偏向高频模板，KL 只会让模型「稳定地」偏向那些模板——坍塌会慢一点，但方向不变（代码 ④ 练习 2 的 `DECAY = 0.95` 就是这个效果）。
**正确做法**：KL 是「刹车」，不是「方向盘」。防坍塌要靠**数据侧的多样性**（去重 + 真人锚点 + 覆盖度监控），两者必须同时上。

### 误区 6：把「递归」当成「无限自我提升」

**错误做法**：认为只要让模型自己改自己，能力就会指数上升，于是把资源全压在「自动进化」上。
**为什么错**：递归放大的是效率，不是正确性；而且每一层都有硬上限——L1 受上下文窗口限制、L3 受数据分布限制（代码 ③ 里难题永远采不到）、L4 受验证成本限制。
**正确做法**：把 RSI 当成「加速器」而不是「永动机」。工程上更划算的组合是：**人类负责定义目标和验证器，模型负责在既定框架内做大规模搜索**——也就是「人在环上的半自动进化」。

### 误区 7：只用「最终答案对不对」筛选自生成数据

**错误做法**：STaR 式训练时只比对最终答案，答案对就收。
**为什么错**：模型会「蒙对」——错误推理算出正确答案的概率并不低（尤其是选择题和数值题），把这些轨迹练回去等于往数据里掺毒。
**正确做法**：过程也要校验（关键步骤 / 中间结果 / 单位），或者用「多次采样答案一致」当置信度门槛；宁可数据少一点，也别让错误推理进训练集。

## 动手练习

四道题由易到难，建议先自己写，再看提示。答案不提供——这些题的乐趣就在于你会踩到坑。

### 练习 1（概念题）：给下面四个系统各贴一个「层级标签」，并说明理由

- A：一个客服 Agent 每次回答前先自问「我有没有漏掉用户的需求」，不满意就重写。
- B：一个代码 Agent 发现自己的 `search` 工具太慢，于是给自己写了个带缓存的版本。
- C：一个数学 Agent 把自己做对的解题过程收集起来，定期微调自己。
- D：一个训练框架让模型自己提出「新的采样温度调度策略」，跑小规模实验后择优采纳。

**考察点**：区分 L1/L2/L3/L4 的判据是「改的对象」，不是「听起来有多高级」。**提示**：问自己一句「这次改动之后，什么东西持久地变了？」

### 练习 2（改代码）：在代码 ① 里加一条「分数相同则比多样性」的平局规则

要求：`push_archive` 收到同分候选时，优先收「与已有解距离更大」的那个。
**提示**：把排序 key 从 `-verifier(...)` 改成一个元组 `(-score, -min_distance)`，其中 `min_distance` 是该候选与 archive 中已有解的最小距离。**考察点**：理解「多样性不是一个事后过滤器，而可以是选择规则本身的一部分」。**预期现象**：archive 的两两平均差会上升，翻过山谷的轮次可能提前。

### 练习 3（改代码）：构造一个「信号饱和」的奖励黑客变体

要求：基于代码 ②，把 `judge_score` 改成分段函数——误差小于 0.3 就给满分 1.0，否则按原来的公式算。也就是让 judge 有一个「宽容平台」。
**提示**：`if abs(a - (t + BIAS)) < 0.3: total += 1.0`。**考察点**：观察「平台」出现后，爬山会怎么走（提示：它会走进平台内部乱漂）。**预期现象**：自评分更快饱和，真实正确率的下滑更早开始——**饱和的软信号比有偏差的软信号更危险**。

### 练习 4（设计题）：给你的 RSI 系统写一份「安全清单」

场景：你要上线一个「每周自动改自己代码」的内部 Agent（改的是工具实现，不改模型权重）。请列出至少 6 条你必须做的防护措施，并说明每条防的是哪种失败。
**提示**：从四个角度想——执行环境（沙箱）、判断依据（评估 + 留出集）、决策权限（谁能批准）、事故应对（回滚 + 审计）。**考察点**：能不能把 (g) 的四根保险丝落到具体工程动作上。**加分项**：说出「哪一条是不可妥协的」（通常是沙箱和回滚——没有它们，其余措施都建立在「不会出事」的假设上）。

## 面试考点

**Q1：什么是递归自我改进（RSI）？和自我纠错、自我改进的区别在哪？**
A：自我纠错是当场改这一次的输出；自我改进是把经验沉淀下来让下次起点更高；RSI 是**改进的对象里包含了「改进器」本身**——改进的产物会改变下一轮改进的能力，从而带来复利（也可能是加速的崩坏）。判据两条：① 改进的产物是否改变了下一轮改进的能力；② 这个过程是否自动发生、不需要人类重新设计流程。

**Q2：RSI 的四层载体分别是什么？**
A：L1 上下文层（Reflexion / Self-Refine，改这一轮的输入，不改参数）；L2 脚手架／代码层（Gödel Machine 思想 → Gödel Agent → Darwin Gödel Machine → AlphaEvolve，改 Agent 的代码与工具）；L3 权重层（STaR、Self-Rewarding LM、Absolute Zero、R-Zero、RLVR，把改进写进参数）；L4 算法／超参层（Self-Developing 类工作，改训练算法与课程）。层次越往下，收益越持久、验证越贵、错误越晚暴露。

**Q3：Reflexion 和 Self-Refine 有什么区别？各自的局限？**
A：Self-Refine 是同一轮内的「生成→自评→自改」，多轮迭代但不跨界；Reflexion 是任务失败后写语言化反思、存进记忆、再用同一模型重试，反思能跨任务复用。共同局限是三件事：反思占上下文预算、反思质量受模型自身洞察力限制、不改变权重所以能力上限不变。

**Q4：Gödel Machine 的思想是什么？为什么没法直接实现？**
A：它设想机器可以改写自己的任何部分（包括改写自己的代码），但**只有当它能形式化证明「这次改写会提升期望效用」时才执行**——事前证明换取了安全性。直接实现不了的原因是对真实系统做形式化效用证明几乎不可行：你无法为「这段代码会让模型更聪明」写一个可判定的证明。工程上的降级路线是「用实测指标代替证明」，这就是 Gödel Agent 的思路。

**Q5：Darwin Gödel Machine 的核心贡献是什么？archive 为什么重要？**
A：核心是把「自改代码」放进一个达尔文式的外部循环：变异（模型改写自身代码）→ 评估（跑编程基准）→ **档案存档（保留 top-k 且彼此差异大的版本）** → 再变异。archive 重要的原因是防止早熟收敛：「只留最高分」会让搜索坍缩到一个小山头，而保留「暂时分数不高但差异大」的版本，等于为未来的跳跃保留了期权。代码 ① 里贪心卡在 105/110，archive 爬到 110/110，差别几乎全部来自这一个设计。

**Q6：AlphaEvolve 和 DGM 有什么相同点、有什么不同？**
A：相同点：都是「生成候选程序 + 自动评估器打分 + 进化筛选」，都靠种群 + 多样性避免早熟。不同点：DGM 优化的对象是「Agent 自己的代码库」（自指），AlphaEvolve 优化的对象是「某个待优化的算法/程序」（不自指，但可以优化训练/调度这类基础设施代码）。一句话：DGM 是在改自己，AlphaEvolve 是在改别人给的问题。

**Q7：STaR 是怎么工作的？它天然有什么缺陷？**
A：采样带推理过程的解答 → 只保留最终答案正确的 → 当 SFT 数据练回去 → 迭代。它还允许「rationalization」：全错时把正确答案当提示，让模型反推一份过程。天然缺陷是「难题采样不到」：难题的正确率极低，在采样预算内几乎采不到正例，于是训练集里难题占比越来越小，模型越训越偏科——代码 ③ 里 pass@1 涨到 0.825 就基本停了，正是这个原因。

**Q8：Self-Rewarding LM 为什么能 work？它的风险是什么？**
A：它让模型自己当 judge 给自己的候选打分，构造偏好对做 DPO，迭代后「答题能力」和「判卷能力」互相促进。能 work 的前提是模型已经具备不错的判断力（judge 能力 ≥ 生成能力的一小截），且指令遵循约束还在。风险是**优化器与评估器同源**：偏差会共振，模型学会的是「让自己满意」，不是「真的做对」（代码 ② 的完整演示）。

**Q9：Absolute Zero 和 R-Zero 各自解决了什么？**
A：Absolute Zero 追求「零外部数据」——模型自己出题（可被代码执行器检验的任务）、自己解题，只用执行器返回的对错当奖励，并提出「可学性」奖励鼓励出「难度刚好在能力边界」的题。R-Zero 把这件事拆成两个角色协同进化：challenger 出贴合当前难度的题、solver 解题，形成自动升级的课程。两者共同的依赖是**独立的验证器**（执行器），主要风险是题目分布单一化或课程失控。

**Q10：什么是 RLVR？为什么它被认为是 L3 的关键？**
A：RLVR（可验证奖励强化学习）是用「能被程序判定的奖励」做 RL——单测通过、答案精确匹配、格式合法、环境状态达成。它关键是因为把「奖励可信度」从软信号提升到了硬信号，让自我训练不再必然滑向奖励黑客；DeepSeek-R1 是把它推向推理任务的代表实践，常用 GRPO 优化。局限是它只能覆盖「可被程序判定」的任务。

**Q11：什么是奖励黑客？怎么防？**
A：优化器找到了评分函数的漏洞，而不是实现真正目标——古德哈特定律的工程版本：「指标一旦变成目标，就不再是好指标」。典型形态：长度偏好导致啰嗦、judge 与策略同源导致自我模仿、固定测试集导致面向测试编程、格式分权重高导致重形式轻内容。防线：硬 verifier 优先、留出集隔离、行为多样性监控、多裁判投票、定期人工校准与红队。

**Q12：什么是模型坍塌？为什么递归训练会加剧它？**
A：Shumailov 等 2023 年在《The Curse of Recursion》里系统讨论的现象：当训练数据主要来自模型自己（或同代模型）的产出，数据分布会一轮轮丢掉尾部——先是罕见样本消失（早期坍塌），再是分布整体漂移、方差收缩（晚期坍塌），最后模型只会重复自己。递归训练加剧它的原因是**每一轮的损失都会被下一轮继承并放大**：这一轮饿死的模板，下一轮更不可能被生成。对策是固定比例的真人数据锚点、去重、覆盖度监控（代码 ④ 的完整演示）。

**Q13：如果面试官问「你怎么判断一次自我改进是真变强还是自我欺骗」，你怎么答？**
A：分四步答：① **看信号**（用的是硬 verifier 还是软 judge，软信号必须配留出集）；② **看分布**（提升是不是只在训练分布的头部，尾部/罕见样本有没有退化）；③ **看外推**（在没见过的任务族、留出集、真实流量上是否也提升）；④ **看可逆性**（能不能一键回滚，改动是否有审计留痕）。再补一句：**真正的判据不是「分数涨了」，而是「分数涨的同时，行为多样性没有塌」**——这句话能把大部分候选人区分开。

**Q14：RSI 目前在工业界的落地程度如何？**
A：L1 全面落地（反思、示例缓存），L2 在代码 Agent 与算法发现上有产品与研究原型（Gödel Agent 类框架、AlphaEvolve 类系统），L3 在大规模使用（STaR 系自蒸馏、RLVR + GRPO），L4 基本停留在研究与半自动形态（模型提方案 + 人类审批 + 小规模 A/B）。原因是验证成本随层级陡增，而错误可见性随层级陡降——L4 改坏了可能三个月后才发现。

**Q15：什么是「观测侧进化」？它和「改提示 / 改模型」有什么本质区别？**

A：RSI 循环有三段：干预 → 训练 → 观测。改提示/改模型动的是前两段，AdaObs 这类工作动的是第三段——**锚点目标不变，但「怎么看」可以变**（细化诊断粒度、升难度、加相关任务）。本质区别在于：它承认「聚合分不再提供信息」这种情况，并把「观测分辨率」当成一等公民来优化。面试时可以补一句边界：观测可以自适应，但**锚点不能动**，否则就成了换考卷。

**Q16：技能级 RSI 和权重级 RSI 怎么选？**

A：看三个维度：①**验证成本**——技能可以单测（跑一遍就知道行不行），权重级要训练+评估；②**可回滚性**——技能条目可以删/改/版本化，权重回滚代价大；③**收益持久性**——权重改动更持久，但错误也更隐蔽。工程实践的顺序通常是：先做技能级（L2）拿到快速收益，再把稳定下来的技能用蒸馏/RL 固化进权重（L3）。这正是 System-One RSI「把慢思考压成肌肉记忆」的思路。

**Q17：2026 年了，为什么还在讨论「RSI 平台期」？平台期可能有哪些来源？**

A：至少三类：①**观测耗尽**——聚合分不再提供 actionable 信息（AdaObs 的论点）；②**多样性耗尽**——archive/数据分布收缩，搜索空间被自己关掉（本章代码 ④ 的坍塌曲线）；③**算力/采样耗尽**——rollout 不够，难题采不到（STaR 的经典上限）。对策分别是对观测做自适应、对多样性设闸门、对 rollout 做扩容与异步（Beam 报告那条曲线）。答题时先反问「你说的平台期是在哪个锚点、用了多少 rollout、多样性指标如何」，立刻就显出层次。

**Q18：Agent 的记忆为什么被当作 RSI 的「基质」而不是普通组件？**

A：因为记忆决定了「改进能不能跨会话累积」。没有记忆，每一轮改进都要从零重来；有了记忆，经验才能沉淀成可复用的资产——这就是「基质」的含义。代价是风险也沉淀：自我幻觉回灌会变成记忆污染，而且更难清除。所以记忆写入必须有闸门（去重、一致性校验、时间戳、可回滚），这也把第 9 章的数据飞轮和第 11 章的 RSI 闭环接在了一起。

## 📡 2026-11 前沿补遗：RSI 的四个新旋钮

> 本章的主体框架建立在 2025 年之前公开的工作上。2026 年 11 月的 arXiv 最新批次里，RSI 这条线又长出了四个值得补进框架的方向——它们不推翻 L1–L4，而是提醒你：**只改「模型/提示」是不够的，观测、技能、记忆、规模同样是可优化的对象。**

### (a) 观测侧进化：平台期也可能是「观测耗尽」

后训练 RSI 的循环可以拆成三段：**干预 → 训练 → 观测（observation）**。绝大多数实现只用锚点 benchmark 的**聚合分**作为唯一观测，于是早期涨得快、之后卡住，然后反复探索同一片区域。

Sage-Evolve-Preview 提出的 AdaObs（adaptive observation）指出了另一种解释：**当聚合分不再提供 actionable 的信息时，平台期反映的是「观测耗尽」，而不是「能力耗尽」。** 做法是锚点目标保持不变、但观测随平台期自适应——细化诊断粒度、提高难度、补充相关任务；验证过的增量再融合回训练，而每个 checkpoint 始终回到锚点上评估。报告里 8 个独立锚点（长上下文阅读、指令跟随）在**匹配的数据预算**下平均分从 66.4 提到 73.7，优于「只观测聚合分」的基线，并在独立的 PostTrainBench 七任务上延续了增益。

对应到本章：这仍然是 **L2（harness）**，但动手位置从「改提示/改代码」变成了「改评估的观测方式」。它并不违反附录 C 的原则——信号强度必须随改进深度单调递增；AdaObs 提升的是信号的**分辨率**，而不是把软信号伪装成硬信号。

### (b) 技能级 RSI：把慢思考压成肌肉记忆

两条互补的路线：

- **System-One RSI**：推理模型「理解强、但亚秒级控制弱」。它用 guided practice 把 System-2 的慢推理蒸馏成 System-1 的快速反应（肌肉记忆），再把改进后的行为反哺回 RSI 循环——交互式 Agent（机器人操控、实时工具调用）受益最大。
- **Evolve Where It Matters**：把 **Agent 的技能库（skill library）** 本身当作可进化对象，用「覆盖度」而不是「平均分」决定往哪里进化——避免所有技能都往已经很强的那几条上堆。

对应到本章：这是 L2 的具体形态从「改代码」扩展到「改技能条目」。回到第 9 章的「四种更新载体」，技能（Skill）这一层终于有了自己的自动化进化闭环。

### (c) 记忆作为 RSI 的基质：从「存在哪」到「怎么折叠」

- **MemFolder**：结构引导的**记忆折叠**——不是把历史压成一段摘要，而是按结构把记忆折成可检索的层级，让「记住什么 / 忘记什么」变成可控操作；
- **ChronoDossier**：时间感知的持久记忆（知道哪条记忆是什么时候、在什么情境下形成的）；
- **STARK**：面向个性化 Agent 的记忆体系。

对应到本章：L1 的上下文改进需要长期基质，而记忆一旦成为基质，**风险也跟着转移**——自我幻觉回灌会变成「记忆污染」，比一次性的错误输出更难清除。所以第 9 章的「数据闸门」要延长到记忆写入侧。

### (d) 跨域扩散 + 规模化：两条腿一起跑

- **跨域扩散**：HandRSI 把「视频手部姿态估计器 + 可执行推理 harness」做成递归自我改进系统（harness 与估计器联合进化）；InteractVLA 让 VLA 可在线调控后做 RSI（LIBERO-Plus 99.4%）。RSI 已经不再是 LLM 专属范式。
- **规模化**：Beam Model Technical Report（2026-11）给出另一条曲线的证据——501.3B 总参 / 23.4B 激活的稀疏 MoE，在 10.5K 张 GB300 上跑异步 RL，4 周生成 1 亿+ rollouts，报告称 RL 算力继续加、能力没有平台期迹象。

把 (a) 和 (d) 放在一起看，就能回答面试最爱问的那句「RSI 会不会撞墙」：**先分清撞的是哪面墙**——观测墙（(a)）、多样性墙（本章 archive/闸门）、还是算力墙（(d)）。三面墙的对策完全不同。

| 旋钮 | 改什么 | 对应层级 | 验证成本 | 典型失败模式 |
| --- | --- | --- | --- | --- |
| 观测侧进化（AdaObs） | 评估的粒度/难度/任务集 | L2 | 中（要防过拟合观测） | 观测越调越像训练集 |
| 技能级进化 | 技能库条目与组合 | L2 | 中（技能可单测） | 技能爆炸、互相冲突 |
| 记忆折叠 | 长期记忆的结构与写入 | L1 | 低–中（可回滚） | 记忆污染、隐私泄漏 |
| 规模化 RL | rollout 数量与异步程度 | L3 | 高（算力+稳定性） | 奖励黑客被放大、成本失控 |

## 附录 A：本章名词速查表

术语第一次出现时我们都给了白话解释，这一节把它们收在一张表里，考前 10 分钟扫一遍最划算。

| 名词 | 中文 / 英文 | 大白话 | 类比 |
| --- | --- | --- | --- |
| RSI | 递归自我改进（Recursive Self-Improvement） | 改进的对象包含了「改进器」本身，于是进步开始复利 | 不只抄错题，还改进了「抄错题的方法」 |
| verifier | 验证器 | 用程序判定对错的硬信号：跑测试、比答案 | 体重秤 |
| LLM-as-judge | 大模型当裁判 | 用一个模型按评分细则给输出打分 | 让同学给你作文打分 |
| reward hacking | 奖励黑客 | 优化器找到了评分函数的漏洞，而不是真的做对 | 为了体检合格前一天不吃饭 |
| model collapse | 模型坍塌 | 递归吃自己产出的数据，分布一轮轮丢掉尾部 | 只吃剩菜，越吃越营养不良 |
| archive | 档案库 | 保留一批「分数不一定最高但彼此不同」的解 | 不只留一道招牌菜的后厨 |
| diversity gate | 多样性闸门 | 新解必须和已有解差得够远才收 | 乐队招人不要五个主唱 |
| archive elitism | 精英保留 | 每代把最好的几个原样带进下一代 | 保留老员工的核心手艺 |
| self-distillation | 自蒸馏 | 模型用自己的（被筛过的）输出当训练数据 | 自己出卷自己批，把满分卷再抄一遍 |
| self-play | 自博弈 | 两个角色（出题/解题）互相喂招一起变强 | 找个陪练，陪你练到两个人都变强 |
| curriculum | 课程 | 按难度自动排序的一组训练任务 | 私教按你的水平排训练计划 |
| RLVR | 可验证奖励强化学习 | 用能被程序判定的奖励做 RL | 用计时器练跑步，而不是凭感觉 |
| KL constraint | KL 约束 | 限制新策略不要离旧策略太远 | 搬家不要一次搬出三个省 |
| held-out set | 留出集 | 从不参与训练的评估数据 | 高考卷不能提前泄露 |
| sandbox | 沙箱 | 限制网络/文件/时长的隔离执行环境 | 装修时的围挡 |
| capability gate | 能力门槛 | 改动超过阈值必须人类批准 | 动承重墙要报批 |
| Goodhart's law | 古德哈特定律 | 指标一旦成为目标，就不再是好指标 | 按销量榜点外卖，未必合你口味 |

## 附录 B：本章引用到的论文与系统清单

> 说明：本节按「年份 + 一句话定位」引用，不编造论文标题与编号。凡是细节（具体数字、实现方式）无法从公开渠道确认的，一律标注「工程实现，详见官方仓库」，请以原文与官方仓库为准。

| 名称 | 年份 | 一句话定位 | 本章用它说明什么 | 确信度 |
| --- | --- | --- | --- | --- |
| Gödel Machine | 2003 | Schmidhuber 提出的「可证明有益的自我改写」理论构想 | L2 的思想源头；「事前证明」路线的代表 | 高 |
| STaR | 2022 | 自生成推理轨迹、只留答对的做自蒸馏 | L3 的自蒸馏原型；难题采不到的上限 | 高 |
| Reflexion | 2023 | 用语言化反思 + 记忆重试来改进 Agent | L1 的代表；反思能跨任务复用 | 高 |
| Self-Refine | 2023 | 同一模型自评自改、多轮迭代 | L1 的代表；「判断比生成容易」 | 高 |
| The Curse of Recursion | 2023 | Shumailov 等关于模型坍塌的系统讨论 | 递归训练丢尾部的机制 | 高 |
| Gödel Agent | 2024 | 运行时改写自身代码、用实测替代证明 | L2 的工程化降级；真正踩进自指 | 高（细节详见官方仓库） |
| Self-Rewarding LM | 2024 | 模型自当 judge 构造偏好对、迭代 DPO | L3 的自奖励；偏差共振风险 | 高 |
| RLVR（可验证奖励强化学习） | 2024–2025 | 一类用程序判定奖励做 RL 的做法 | L3 的主流工程路线；DeepSeek-R1 为代表实践 | 高（作为一类做法而非单篇论文） |
| Darwin Gödel Machine | 2025 | 自改代码库 + archive 保留多样版本 | archive 为什么是核心贡献 | 高（具体基准数字以论文为准） |
| AlphaEvolve | 2025 | 进化搜索程序、用自动评估器筛选 | L2 的算法发现形态；矩阵乘法等案例 | 高（具体数值详见官方博客/仓库） |
| Absolute Zero | 2025 | 零外部数据的自出题自解题 + 可学性奖励 | L3 的「断奶」路线 | 高（细节详见官方仓库） |
| R-Zero | 2025 | challenger / solver 双角色协同进化 | 课程自动化的实现方式 | 中（工程实现，详见官方仓库） |
| Self-Developing | 2025 | 让模型合成「改进算法」的思路 | L4 的代表方向 | 中（工程实现，详见官方仓库） |
| AlphaZero | 2017 | 自我对弈的经典范式 | 自博弈思想源头 | 高 |

| Sage-Evolve / AdaObs | 2026 | 后训练 RSI 的观测侧自适应（arXiv:2611.26132） | 「平台期=观测耗尽」的机制 | 高 |
| System-One RSI | 2026 | 用 guided practice 把慢思考蒸馏成亚秒级反应（arXiv:2611.26007） | 技能级 RSI / 肌肉记忆 | 高 |
| Beam Model Technical Report | 2026 | 501.3B 稀疏 MoE + 10.5K GB300 异步 RL（arXiv:2611.26358） | 规模化旋钮；RL 算力未见平台期 | 高 |
| MemFolder | 2026 | 结构引导的 Agent 记忆折叠（arXiv:2611.26427） | 记忆作为 L1 基质 | 高 |
| InteractVLA | 2026 | 可在线调控的 VLA + RSI（LIBERO-Plus 99.4%，数字取自摘要） | RSI 跨域扩散到机器人 | 中（细节以论文为准） |
| HandRSI | 2026 | 手部姿态估计的 RSI（cs.CV，2026-11） | RSI 跨域扩散 | 中（细节以论文为准） |
| Evolve Where It Matters | 2026 | 覆盖率引导的 Agent 技能自进化（cs.CL，2026-11） | 技能库作为可进化对象 | 中（细节以论文为准） |
| ChronoDossier / STARK | 2026 | 时间感知持久记忆 / 个性化 Agent 记忆 | 记忆基质的两条工程路线 | 中（细节以论文为准） |

**如果你只想记三个年份**：2003（敢想）→ 2024（敢改、敢判）→ 2025（敢养种群、敢断奶）。这三个节点正好对应本章「三次胆子变大」那条时间线。

## 附录 C：四层 × 信号 的组合矩阵

同一个层级，配上不同的信号，结果可能完全相反。这张矩阵是本章所有结论的浓缩版：

| | 硬 verifier（单测 / 精确匹配） | 规则校验（schema / 格式） | LLM-judge | 模型自评分 |
| --- | --- | --- | --- | --- |
| **L1 上下文** | 可靠：反思「哪一步错了」有据可依 | 好用：格式类问题直接修 | 常见：写作类任务的主流做法 | 少见：容易陷入自我循环 |
| **L2 代码** | 最理想：测试通过就是通过 | 补充：语法/规范检查 | 偶用：评估「代码可读性」 | 危险：会写出「自己觉得优雅」的烂代码 |
| **L3 权重** | 黄金组合：RLVR / STaR 的前提 | 辅助：过滤格式错误样本 | 高风险：代码 ② 演示的就是这条路 | 最高风险：Self-Rewarding 的软肋 |
| **L4 算法** | 必须：没有可信评估就不该碰算法层 | 辅助：约束生成的算法代码可运行 | 不可用：无法判断「算法是否更好」 | 禁止：等于让噪声决定训练方法 |

> 💡 **解读**：这张矩阵的对角线（左上到右下）是「可靠区」——层级越深，越必须配硬信号；右上角那一块是「危险区」——层级越深还配软信号，等于在悬崖边蒙眼开车。面试时如果被问「你会怎么设计奖励」，把这张矩阵的语言说出来（「我的原则是：信号强度必须随改进深度单调递增」），比背任何单个方法都更有说服力。

## 小结与下一章预告

这一章是全书唯一一章「主角不是阿零的能力，而是阿零的进化方式」。回头看，我们做了四件事：

第一，立了框架：**改进的对象越往下沉一层，收益越持久，验证越贵，错误越晚暴露**——L1 改草稿、L2 改代码、L3 改权重、L4 改算法。这条「深度—代价」曲线，是判断任何自我改进系统成熟度的第一把尺子。

第二，拆了信号：**自我改进的上限由评估的上限决定**。可执行 verifier 是最硬的锚，LLM-judge 是弱的，模型自评分是最弱的——而代码 ② 用一组数字把「自评分 0.589→0.910、真实正确率 0.917→0.100」这个过程钉在了你脑子里。记住那句总结：**优化器永远会优化你给它的东西，包括它的缺陷。**

第三，装了安全带：多样性（archive / 温度 / KL）、防坍塌（真人锚点 / 去重 / 覆盖度监控）、防黑客（留出集 / 行为监控 / 红队）、安全边界（沙箱 / 审计 / 门槛 / 回滚）。四根保险丝对应四种翻车方式，缺一根，那圈闭环就从「进化」变成「事故预演」。

第四，亲手跑了四个零件：archive 搜索证明了多样性值多少钱（105 → 110）、奖励黑客演示了软信号的代价、STaR 演示了自蒸馏的真实曲线与其天花板、数据闸门演示了坍缩与止损。

**下一章预告**——严格说，本章已经是「12-智能体开发」模块的最后一章（第 10 章含全书总结，本章是它的「元」补完）。所以「下一章」不再是书里的下一章，而是**你的下一站**，有三条路可选：

1. **补地基（面试最常被追问）**：本章的 L3/L4 全部建立在强化学习与训练范式上，建议系统过一遍 [../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb](../04-LLM大模型/教学/21-PPO与GRPO手推.ipynb)（PPO/GRPO 手推）、[../07-强化学习/教学/06-PPO与GRPO.ipynb](../07-强化学习/教学/06-PPO与GRPO.ipynb)（框架视角），再回看第 8 章 [08-Mid-training-SFT与RL.md](08-Mid-training-SFT与RL.md) 会更顺。
2. **补评估（决定你敢不敢做 RSI）**：回看 [07-评估与Pass-at-k.md](07-评估与Pass-at-k.md)，重点是自己实现一遍无偏 pass@k 与 bootstrap 置信区间——代码 ① 里「多 seed 看方差」的习惯，来源就是那一章。
3. **补工程（落地最缺的）**：回看 [09-轨迹学习与持续进化.md](09-轨迹学习与持续进化.md) 的数据飞轮与回滚，把本章的四根保险丝装到那条飞轮上——**飞轮 + 安全带，才是一个能上线的自我改进系统。**

最后留一句话给正在准备面试的你：**RSI 不是「让模型自己变强」的魔法，而是「把改进的权限和验证的责任一起交出去」的工程决策。** 权限交得越多，验证就必须做得越扎实。如果面试官问你「你敢让 Agent 自己改自己吗」，最好的回答不是「敢」或「不敢」，而是：**「在什么沙箱里、用什么验证器、谁来审批、出事怎么回滚」——把这四件事说清楚，才叫真的敢。**